# 01a: Data Extraction

Extract bed occupancy data from **COVID-19 Reported Patient Impact and Hospital Capacity by Facility** dataset for a selected hospital and prepare for forecasting analysis.

**Data Source:** HealthData.gov - COVID-19 Reported Patient Impact and Hospital Capacity by Facility  

https://healthdata.gov/Hospital/COVID-19-Reported-Patient-Impact-and-Hospital-Capa/anag-cw7u/about_data

**Period:** Automatically derived from the selected hospital's quality data availability  
**Hospital:** Configurable (default: ADVENTHEALTH ORLANDO, Orlando, FL)  
**Data Type:** Weekly bed occupancy (7-day averages)

**Note:**
- This dataset contains weekly aggregated bed occupancy and capacity metrics
- Data is already weekly (not daily), so no aggregation needed
- Bed occupancy represents average daily occupied beds over each 7-day period
- Dates are real (not deidentified), so no anchor year alignment needed
- **Date Range:** The start/end dates are automatically derived from the selected hospital's quality data (weeks with valid occupancy values), not from the overall dataset dates
- **ADVENTHEALTH ORLANDO:** Quality data runs from 2020-07-19 to 2024-04-21 (197 weeks). Early weeks (2020-03-22 to 2020-07-12) were excluded due to missing occupancy data.

**Setup:**
1. Ensure `COVID-19_Reported_Patient_Impact_and_Hospital_Capacity_by_Facility.csv` is in `data/raw/` directory
2. The notebook will filter to the selected hospital and process the data automatically
3. External data (weather, flu) will be fetched for the hospital's specific date range

**Notebook Structure:**
- **Part 1: Hospital Data Processing** - Load COVID CSV, filter to selected hospital, extract bed occupancy
- **Part 2: External Data Extraction** - Download weather and flu data for hospital location (date range from hospital data)
- **Part 3: Verification & Saving** - Verify all data and save cleaned files


In [16]:
import pandas as pd
import os
import warnings
import numpy as np
warnings.filterwarnings('ignore')

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def ensure_package(package_name, import_name=None):
    """Install package if not available"""
    import_name = import_name or package_name.split('.')[0]
    try:
        __import__(import_name)
        print(f"✓ {package_name} already available")
        return True
    except ImportError:
        print(f"📦 Installing {package_name}...")
        os.system(f"pip install {package_name} -q")
        print(f"✓ {package_name} installed")
        return True

def save_to_locations(df, filename, data_dir, drive_dir=None):
    """Save dataframe to local and optionally drive location"""
    local_path = os.path.join(data_dir, filename)
    df.to_csv(local_path, index=False)
    print(f"✓ Saved to: {local_path}")

    if drive_dir:
        drive_path = os.path.join(drive_dir, filename)
        df.to_csv(drive_path, index=False)
        print(f"✓ Saved to: {drive_path}")

    return local_path

def align_data_to_dates(source_df, target_dates, tolerance_days=3, verbose=True):
    """Align source data to target dates using nearest matching

    Args:
        source_df: DataFrame with 'date' column to align
        target_dates: DatetimeIndex of target dates to align to
        tolerance_days: Maximum days difference for matching (default 3)
        verbose: Print diagnostic information (default True)

    Returns:
        DataFrame indexed by target_dates with aligned data
    """
    if len(source_df) == 0 or len(target_dates) == 0:
        if verbose:
            print(f"  ⚠️  align_data_to_dates: Empty input (source={len(source_df)}, target={len(target_dates)})")
        return pd.DataFrame()

    source_indexed = source_df.set_index('date')
    aligned_data = []
    used_dates = set()

    # Diagnostic counters
    exact_matches = 0
    nearest_matches = 0
    no_matches = 0
    match_distances = []

    if verbose:
        print(f"\n  📊 ALIGNMENT DIAGNOSTICS:")
        print(f"     Source data: {len(source_indexed)} records")
        print(f"     Source date range: {source_indexed.index.min().date()} to {source_indexed.index.max().date()}")
        print(f"     Source day of week: {source_indexed.index[0].strftime('%A')}")
        print(f"     Target dates: {len(target_dates)} dates")
        print(f"     Target date range: {target_dates.min().date()} to {target_dates.max().date()}")
        print(f"     Target day of week: {target_dates[0].strftime('%A')}")
        print(f"     Tolerance: ±{tolerance_days} days")

    for target_date in target_dates:
        # Try exact match first
        if target_date in source_indexed.index and target_date not in used_dates:
            row = source_indexed.loc[target_date].to_dict()
            used_dates.add(target_date)
            exact_matches += 1
            match_distances.append(0)
        else:
            # Find nearest within tolerance
            available = source_indexed[~source_indexed.index.isin(used_dates)]
            if len(available) > 0:
                nearest = available.index[(abs(available.index - target_date)).argmin()]
                distance = abs((nearest - target_date).days)
                if distance <= tolerance_days:
                    row = source_indexed.loc[nearest].to_dict()
                    used_dates.add(nearest)
                    nearest_matches += 1
                    match_distances.append(distance)
                else:
                    row = {col: np.nan for col in source_indexed.columns}
                    no_matches += 1
            else:
                row = {col: np.nan for col in source_indexed.columns}
                no_matches += 1

        row['date'] = target_date
        aligned_data.append(row)

    if verbose:
        print(f"     ---")
        print(f"     Exact matches: {exact_matches} ({exact_matches/len(target_dates)*100:.1f}%)")
        print(f"     Nearest matches (within tolerance): {nearest_matches} ({nearest_matches/len(target_dates)*100:.1f}%)")
        print(f"     No matches (NaN filled): {no_matches} ({no_matches/len(target_dates)*100:.1f}%)")
        if match_distances:
            print(f"     Match distance stats: min={min(match_distances)}, max={max(match_distances)}, avg={np.mean(match_distances):.1f} days")
        if no_matches > 0:
            print(f"     ⚠️  {no_matches} dates could not be matched - will need interpolation")

    return pd.DataFrame(aligned_data).set_index('date') if aligned_data else pd.DataFrame()

def get_seasonal_flu_activity(date):
    """Create seasonal flu activity proxy based on month"""
    month = pd.to_datetime(date).month
    activity_map = {
        12: 3.0, 1: 3.0, 2: 3.0,  # Winter - peak
        3: 2.0, 4: 2.0, 5: 2.0,   # Spring
        6: 0.5, 7: 0.5, 8: 0.5,   # Summer - minimal
    }
    return activity_map.get(month, 1.0)  # Fall default

def validate_dataframe(df, name, required_cols=None, date_col=None):
    """Run common validation checks"""
    issues = []

    if required_cols:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            issues.append(f"Missing columns: {missing}")

    if date_col and date_col in df.columns and len(df) > 1:
        diffs = df[date_col].diff().dt.days
        invalid = diffs[(diffs.notna()) & ((diffs < 6) | (diffs > 8))]
        if len(invalid) > 0:
            issues.append(f"{len(invalid)} date gaps not ~7 days")

    if issues:
        print(f"  ⚠️  {name} validation warnings: {'; '.join(issues)}")
        return False
    else:
        print(f"  ✓ {name} validation passed")
        return True

# Detect environment (Colab vs local)
IS_COLAB = False
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# Mount Google Drive (if using Colab)
DRIVE_ROOT = None
DRIVE_DATA_RAW = None
DRIVE_DATA_EXTERNAL = None

if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_ROOT = '/content/drive/MyDrive/hospital_occupancy_forecasting'
        DRIVE_DATA_RAW = os.path.join(DRIVE_ROOT, 'data', 'raw')
        DRIVE_DATA_EXTERNAL = os.path.join(DRIVE_ROOT, 'data', 'external')
        os.makedirs(DRIVE_DATA_RAW, exist_ok=True)
        os.makedirs(DRIVE_DATA_EXTERNAL, exist_ok=True)
    except Exception as e:
        print(f"⚠️  Could not mount Google Drive: {e}")
        DRIVE_ROOT = None
        DRIVE_DATA_RAW = None
        DRIVE_DATA_EXTERNAL = None

# Setup paths - works for both Colab and local environments
if IS_COLAB:
    PROJECT_ROOT = '/content/hospital_occupancy_forecasting'
else:
    # Local environment: use current working directory
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
    if not os.path.exists(PROJECT_ROOT) or 'hospital_occupancy_forecasting' not in PROJECT_ROOT:
        # Fallback: assume we're in the project root
        PROJECT_ROOT = os.path.abspath('.')

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
DATA_RAW = os.path.join(DATA_DIR, 'raw')
DATA_EXTERNAL = os.path.join(DATA_DIR, 'external')

# Create directories
for path in [DATA_RAW, DATA_EXTERNAL]:
    os.makedirs(path, exist_ok=True)

print("✓ Setup complete")
print(f"  Environment: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"  Project root: {PROJECT_ROOT}")
print(f"  Data (raw): {DATA_RAW}")
print(f"  Data (external): {DATA_EXTERNAL}")
if DRIVE_DATA_RAW:
    print(f"  Google Drive (raw): {DRIVE_DATA_RAW}")
    print(f"  Google Drive (external): {DRIVE_DATA_EXTERNAL}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Setup complete
  Environment: Google Colab
  Project root: /content/hospital_occupancy_forecasting
  Data (raw): /content/hospital_occupancy_forecasting/data/raw
  Data (external): /content/hospital_occupancy_forecasting/data/external
  Google Drive (raw): /content/drive/MyDrive/hospital_occupancy_forecasting/data/raw
  Google Drive (external): /content/drive/MyDrive/hospital_occupancy_forecasting/data/external


# Part 1: ADVENTHEALTH ORLANDO Bed Occupancy Data Processing

Load COVID-19 dataset, filter to ADVENTHEALTH ORLANDO, and extract bed occupancy data (weekly 7-day averages).


## Configuration: Hospital Selection

ADVENTHEALTH ORLANDO is pre-selected as the target hospital for bed occupancy forecasting.


## File Upload (Colab Users)

If you're running this notebook in Google Colab and don't have the COVID-19 CSV file in Google Drive, you can upload it directly here.


In [17]:
# File upload helper for Colab users
# If running in Colab and file not found, this cell will help upload it

COVID_CSV_FILENAME = 'COVID-19_Reported_Patient_Impact_and_Hospital_Capacity_by_Facility.csv'

if IS_COLAB:
    from google.colab import files
    import shutil

    # Check if file already exists
    covid_file_exists = False
    for check_path in [
        os.path.join(DRIVE_DATA_RAW, COVID_CSV_FILENAME) if DRIVE_DATA_RAW else None,
        os.path.join(DATA_RAW, COVID_CSV_FILENAME)
    ]:
        if check_path and os.path.exists(check_path):
            covid_file_exists = True
            print(f"✓ File already exists at: {check_path}")
            break

    if not covid_file_exists:
        print("📤 File not found. Please upload the COVID-19 CSV file.")
        print(f"   File name: {COVID_CSV_FILENAME}")
        print(f"   Expected location: {DATA_RAW}")
        print("\n   Click 'Choose Files' below to upload:")
        uploaded = files.upload()

        # Move uploaded file to correct location
        for filename in uploaded.keys():
            if COVID_CSV_FILENAME.lower() in filename.lower() or filename.endswith('.csv'):
                dest_path = os.path.join(DATA_RAW, COVID_CSV_FILENAME)
                shutil.move(filename, dest_path)
                print(f"✓ File uploaded and moved to: {dest_path}")
                break
        else:
            print("⚠️  Warning: Uploaded file name doesn't match expected name.")
            print(f"   Please ensure the file is named: {COVID_CSV_FILENAME}")
else:
    print("✓ Running locally - file should be in data/raw/ directory")
    print(f"   Expected: {os.path.join(DATA_RAW, COVID_CSV_FILENAME)}")

✓ File already exists at: /content/drive/MyDrive/hospital_occupancy_forecasting/data/raw/COVID-19_Reported_Patient_Impact_and_Hospital_Capacity_by_Facility.csv


In [18]:
# Configuration: Hospital selection
SELECTED_HOSPITAL = 'ADVENTHEALTH ORLANDO'

print(f"Selected hospital: {SELECTED_HOSPITAL}")
print(f"Location: Orlando, FL")
print(f"Target metric: Bed occupancy (weekly 7-day averages)")

Selected hospital: ADVENTHEALTH ORLANDO
Location: Orlando, FL
Target metric: Bed occupancy (weekly 7-day averages)


In [19]:
# Load COVID-19 Hospital Capacity dataset
COVID_CSV_FILENAME = 'COVID-19_Reported_Patient_Impact_and_Hospital_Capacity_by_Facility.csv'

# Search for file in multiple locations
covid_csv_paths = []
if DRIVE_DATA_RAW:
    covid_csv_paths.append(os.path.join(DRIVE_DATA_RAW, COVID_CSV_FILENAME))
covid_csv_paths.append(os.path.join(DATA_RAW, COVID_CSV_FILENAME))

# Also check parent directories (for flexibility)
if not IS_COLAB:
    # Check if file is in current directory or parent
    for check_path in ['.', '..', '../..']:
        potential_path = os.path.join(check_path, 'data', 'raw', COVID_CSV_FILENAME)
        if os.path.exists(potential_path):
            covid_csv_paths.append(os.path.abspath(potential_path))

covid_csv_path = None
for path in covid_csv_paths:
    if path and os.path.exists(path):
        covid_csv_path = path
        break

if covid_csv_path is None:
    error_msg = (
        f"❌ {COVID_CSV_FILENAME} not found!\n"
        f"   Searched in:\n"
    )
    for path in covid_csv_paths:
        error_msg += f"     - {path}\n"
    error_msg += (
        f"\n   Please download from HealthData.gov and place in one of these locations:\n"
        f"     - {DATA_RAW}\n"
    )
    if DRIVE_DATA_RAW:
        error_msg += f"     - {DRIVE_DATA_RAW}\n"
    raise FileNotFoundError(error_msg)

print(f"✓ Found COVID dataset: {covid_csv_path}")
file_size_mb = os.path.getsize(covid_csv_path) / (1024**2)

# If file is local and Drive is available, copy to Drive for persistence
if DRIVE_DATA_RAW and not covid_csv_path.startswith('/content/drive'):
    drive_path = os.path.join(DRIVE_DATA_RAW, COVID_CSV_FILENAME)
    if not os.path.exists(drive_path):
        print(f"\n💾 Copying COVID dataset to Google Drive for persistence...")
        try:
            import shutil
            shutil.copy2(covid_csv_path, drive_path)
            print(f"  ✓ Saved to: {drive_path}")
        except Exception as e:
            print(f"  ⚠️  Could not copy to Drive: {e}")
            print(f"     File will be used from local path only")

print(f"  File size: {file_size_mb:.1f} MB")

# Load the COVID dataset with error handling
print("\n📥 Loading COVID-19 dataset (this may take a moment due to large file size)...")
try:
    # Use chunking for very large files to show progress
    if file_size_mb > 500:  # For files > 500MB, use chunking
        print("  Using chunked reading for large file...")
        chunks = []
        chunk_size = 100000  # Read 100k rows at a time
        total_chunks = None  # Will be estimated after first chunk

        for i, chunk in enumerate(pd.read_csv(covid_csv_path, chunksize=chunk_size,
                                               parse_dates=['collection_week'],
                                               low_memory=False)):
            chunks.append(chunk)

            # Estimate total chunks on first iteration
            if i == 0:
                file_size_bytes = os.path.getsize(covid_csv_path)
                chunk_size_bytes = chunk.memory_usage(deep=True).sum()
                total_chunks = max(1, int((file_size_bytes / chunk_size_bytes) * 1.1))  # 10% buffer

            # Progress indicator with estimated total
            if total_chunks:
                progress_pct = min(100, ((i + 1) / total_chunks) * 100)
                print(f"    Progress: {(i + 1) * chunk_size:,} rows loaded (~{progress_pct:.1f}%)...", end='\r')
            else:
                print(f"    Loaded {(i + 1) * chunk_size:,} rows...", end='\r')

        covid_df = pd.concat(chunks, ignore_index=True)
        print(f"\n✓ Loaded {len(covid_df):,} rows ({len(chunks)} chunks processed)")
    else:
        covid_df = pd.read_csv(
            covid_csv_path,
            parse_dates=['collection_week'],
            low_memory=False
        )
        print(f"✓ Loaded {len(covid_df):,} weekly records")
except Exception as e:
    raise RuntimeError(
        f"❌ Error loading COVID dataset: {e}\n"
        f"   File: {covid_csv_path}\n"
        f"   Please check that the file is not corrupted and is a valid CSV."
    ) from e

# Validate required columns
required_cols = ['collection_week', 'hospital_name', 'state']
missing_cols = [col for col in required_cols if col not in covid_df.columns]
if missing_cols:
    raise ValueError(
        f"❌ Missing required columns in dataset: {missing_cols}\n"
        f"   Available columns: {list(covid_df.columns[:10])}..."
    )

# Validate date column
if covid_df['collection_week'].isna().all():
    raise ValueError("❌ All dates in 'collection_week' are missing!")

print(f"\n📊 Dataset Summary:")
print(f"  Total records: {len(covid_df):,}")
print(f"  Date range: {covid_df['collection_week'].min().date()} to {covid_df['collection_week'].max().date()}")
print(f"  Hospitals: {covid_df['hospital_name'].nunique():,} unique")
print(f"  States: {covid_df['state'].nunique()} unique")
print(f"  Columns: {len(covid_df.columns)}")

# List all columns in the dataset
print(f"\n📋 All Columns in COVID-19 Dataset ({len(covid_df.columns)} total):")
print("  " + "="*70)
for i, col in enumerate(covid_df.columns, 1):
    # Show data type and sample non-null count for each column
    non_null_count = covid_df[col].notna().sum()
    non_null_pct = (non_null_count / len(covid_df)) * 100 if len(covid_df) > 0 else 0
    dtype = str(covid_df[col].dtype)
    print(f"  {i:3d}. {col:<60} [{dtype:<10}] ({non_null_count:,}/{len(covid_df):,} non-null, {non_null_pct:.1f}%)")
print("  " + "="*70)

# Show sample of raw data
print(f"\n📋 Sample of Raw Data (first 3 rows):")
sample_cols = ['collection_week', 'hospital_name', 'state', 'city']
if 'all_adult_hospital_inpatient_bed_occupied_7_day_avg' in covid_df.columns:
    sample_cols.append('all_adult_hospital_inpatient_bed_occupied_7_day_avg')
display_cols = [col for col in sample_cols if col in covid_df.columns]
print(covid_df[display_cols].head(3).to_string(index=False))

✓ Found COVID dataset: /content/drive/MyDrive/hospital_occupancy_forecasting/data/raw/COVID-19_Reported_Patient_Impact_and_Hospital_Capacity_by_Facility.csv
  File size: 685.8 MB

📥 Loading COVID-19 dataset (this may take a moment due to large file size)...
  Using chunked reading for large file...

✓ Loaded 1,045,406 rows (11 chunks processed)

📊 Dataset Summary:
  Total records: 1,045,406
  Date range: 2019-12-29 to 2024-04-21
  Hospitals: 5,017 unique
  States: 56 unique
  Columns: 128

📋 All Columns in COVID-19 Dataset (128 total):
    1. hospital_pk                                                  [object    ] (1,045,406/1,045,406 non-null, 100.0%)
    2. collection_week                                              [datetime64[ns]] (1,045,406/1,045,406 non-null, 100.0%)
    3. state                                                        [object    ] (1,045,406/1,045,406 non-null, 100.0%)
    4. ccn                                                          [object    ] (1,044,004/1,

## Filter to ADVENTHEALTH ORLANDO and Extract Bed Occupancy


In [20]:
# Filter to ADVENTHEALTH ORLANDO
print(f"\n🔍 Searching for hospital: {SELECTED_HOSPITAL}")

# Handle case-insensitive matching
hospital_mask = covid_df['hospital_name'].str.contains(SELECTED_HOSPITAL, case=False, na=False)
adventhealth_df = covid_df[hospital_mask].copy()

if len(adventhealth_df) == 0:
    # Try exact match
    adventhealth_df = covid_df[covid_df['hospital_name'] == SELECTED_HOSPITAL].copy()
    if len(adventhealth_df) == 0:
        # Show similar hospital names for debugging
        similar = covid_df[covid_df['hospital_name'].str.contains('ADVENT', case=False, na=False)]['hospital_name'].unique()
        error_msg = (
            f"❌ Hospital '{SELECTED_HOSPITAL}' not found!\n"
            f"   Found {len(similar)} hospitals with 'ADVENT' in name:\n"
        )
        for i, name in enumerate(similar[:10], 1):
            error_msg += f"     {i}. {name}\n"
        if len(similar) > 10:
            error_msg += f"     ... and {len(similar) - 10} more\n"
        error_msg += f"\n   Tip: Check exact spelling or use one of the hospitals listed above."
        raise ValueError(error_msg)

hospital_name = adventhealth_df['hospital_name'].iloc[0]
city = adventhealth_df['city'].iloc[0] if 'city' in adventhealth_df.columns else 'N/A'
state = adventhealth_df['state'].iloc[0]

print(f"✓ Found: {hospital_name}")
print(f"  Location: {city}, {state}")
print(f"  Records: {len(adventhealth_df):,} weekly records")
print(f"  Date range: {adventhealth_df['collection_week'].min().date()} to {adventhealth_df['collection_week'].max().date()}")

# Key columns for bed occupancy (define BEFORE using them)
occupancy_col = 'all_adult_hospital_inpatient_bed_occupied_7_day_avg'
capacity_col = 'all_adult_hospital_inpatient_beds_7_day_avg'
coverage_col = 'all_adult_hospital_inpatient_bed_occupied_7_day_coverage'

# Check day of week for collection dates
if 'collection_week' in adventhealth_df.columns:
    collection_dates = adventhealth_df['collection_week'].dropna()
    if len(collection_dates) > 0:
        # Get day of week (0=Monday, 6=Sunday)
        day_of_week = collection_dates.dt.dayofweek
        day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

        # Count occurrences of each day
        day_counts = day_of_week.value_counts().sort_index()

        print(f"\n📅 Collection Week Day of Week Analysis:")
        print(f"  Total collection dates: {len(collection_dates):,}")
        for day_num, count in day_counts.items():
            day_name = day_names[day_num]
            pct = (count / len(collection_dates)) * 100
            print(f"    {day_name}: {count} dates ({pct:.1f}%)")

        # Show most common day
        most_common_day = day_counts.idxmax()
        most_common_name = day_names[most_common_day]
        print(f"\n  Most common day: {most_common_name} ({day_counts[most_common_day]} dates, {day_counts[most_common_day]/len(collection_dates)*100:.1f}%)")

        # Show sample dates with their day names
        print(f"\n  Sample dates with day of week:")
        sample_dates = collection_dates.head(5)
        for date in sample_dates:
            day_name = day_names[date.dayofweek]
            print(f"    {date.date()} - {day_name}")

# Show sample of filtered data (before cleaning)
print(f"\n📋 Sample of Filtered Data (before cleaning, first 3 rows):")
sample_cols = ['collection_week', 'hospital_name', 'city', 'state', occupancy_col]
if capacity_col in adventhealth_df.columns:
    sample_cols.append(capacity_col)
if coverage_col in adventhealth_df.columns:
    sample_cols.append(coverage_col)
display_cols = [col for col in sample_cols if col in adventhealth_df.columns]
print(adventhealth_df[display_cols].head(3).to_string(index=False))

# Validate required columns exist
required_cols = [occupancy_col, coverage_col]
missing_cols = [col for col in required_cols if col not in adventhealth_df.columns]
if missing_cols:
    raise ValueError(
        f"❌ Missing required columns: {missing_cols}\n"
        f"   Available columns: {list(adventhealth_df.columns)}"
    )

# Replace missing data codes
print(f"\n🧹 Cleaning data (replacing missing codes)...")
missing_codes = [-999999, -999, '-999,999', '-999999', '-999,999.0']
for col in [occupancy_col, capacity_col]:
    if col in adventhealth_df.columns:
        initial_nulls = adventhealth_df[col].isna().sum()
        # Replace missing codes
        adventhealth_df[col] = adventhealth_df[col].replace(missing_codes, np.nan)
        # Handle comma-separated numbers (convert to numeric)
        # Optimized: Only process non-null values to avoid NaN → 'nan' → NaN conversion
        mask_not_null = adventhealth_df[col].notna()
        if mask_not_null.any():
            # For non-null values, convert to string, remove commas, then to numeric
            adventhealth_df.loc[mask_not_null, col] = (
                adventhealth_df.loc[mask_not_null, col]
                .astype(str)
                .str.replace(',', '', regex=False)
            )
        # Convert entire column to numeric (NaN values will remain NaN)
        adventhealth_df[col] = pd.to_numeric(adventhealth_df[col], errors='coerce')
        final_nulls = adventhealth_df[col].isna().sum()
        if final_nulls > initial_nulls:
            print(f"  {col}: Replaced {final_nulls - initial_nulls} missing codes with NaN")

# Data overview
print(f"\n📊 Data Overview:")
print(f"  Shape: {adventhealth_df.shape[0]:,} rows × {adventhealth_df.shape[1]} columns")
print(f"\n  Key Columns:")
print(f"    • Occupancy: {occupancy_col}")
print(f"    • Capacity: {capacity_col if capacity_col in adventhealth_df.columns else 'N/A'}")
print(f"    • Coverage: {coverage_col}")

# Check missing values
print(f"\n  Missing Values:")
check_cols = [occupancy_col, coverage_col]
if capacity_col in adventhealth_df.columns:
    check_cols.append(capacity_col)
missing = adventhealth_df[check_cols].isnull().sum()
if missing.sum() > 0:
    for col, count in missing.items():
        pct = (count / len(adventhealth_df)) * 100
        print(f"    • {col}: {count:,} ({pct:.1f}%)")
else:
    print("    ✓ No missing values in key columns")

# Quality filter: only weeks with good coverage (≥4 days) and valid occupancy
# Set to False to include all weeks regardless of coverage
APPLY_COVERAGE_FILTER = False  # Set to True to exclude weeks with coverage < 4 days

if APPLY_COVERAGE_FILTER:
    print(f"\n🔍 Applying quality filter (coverage ≥4 days, valid occupancy)...")
    quality_mask = (adventhealth_df[coverage_col] >= 4) & adventhealth_df[occupancy_col].notna()
    adventhealth_clean = adventhealth_df[quality_mask].copy()
else:
    print(f"\n🔍 Applying minimal filter (valid occupancy only, coverage filter disabled)...")
    # Only filter out rows with missing occupancy data
    quality_mask = adventhealth_df[occupancy_col].notna()
    adventhealth_clean = adventhealth_df[quality_mask].copy()

print(f"\n📈 Data Quality Results:")
print(f"  Total weeks: {len(adventhealth_df):,}")
if APPLY_COVERAGE_FILTER:
    print(f"  Quality weeks (coverage ≥4 days): {len(adventhealth_clean):,} ({len(adventhealth_clean)/len(adventhealth_df)*100:.1f}%)")
    print(f"  Excluded weeks: {len(adventhealth_df) - len(adventhealth_clean):,}")
else:
    print(f"  Weeks with valid occupancy: {len(adventhealth_clean):,} ({len(adventhealth_clean)/len(adventhealth_df)*100:.1f}%)")
    print(f"  Excluded weeks (missing occupancy): {len(adventhealth_df) - len(adventhealth_clean):,}")
    print(f"  Note: Coverage filter is disabled - all weeks with valid occupancy are included")

if len(adventhealth_clean) == 0:
    raise ValueError(
        "❌ No quality data found after filtering!\n"
        "   Check that coverage and occupancy columns have valid data."
    )

# Calculate statistics
print(f"\n📊 Bed Occupancy Statistics (quality-filtered):")
occ_stats = adventhealth_clean[occupancy_col].describe()
print(f"  Mean: {occ_stats['mean']:.1f} beds")
print(f"  Median: {occ_stats['50%']:.1f} beds")
print(f"  Std Dev: {occ_stats['std']:.1f} beds")
print(f"  Min: {occ_stats['min']:.1f} beds")
print(f"  Max: {occ_stats['max']:.1f} beds")
print(f"  25th percentile: {occ_stats['25%']:.1f} beds")
print(f"  75th percentile: {occ_stats['75%']:.1f} beds")

# Calculate occupancy rate if capacity is available
if capacity_col in adventhealth_clean.columns and adventhealth_clean[capacity_col].notna().sum() > 0:
    valid_capacity = adventhealth_clean[capacity_col].notna()
    if valid_capacity.sum() > 0:
        adventhealth_clean.loc[valid_capacity, 'occupancy_rate'] = (
            adventhealth_clean.loc[valid_capacity, occupancy_col] /
            adventhealth_clean.loc[valid_capacity, capacity_col]
        ) * 100

        rate_stats = adventhealth_clean['occupancy_rate'].describe()
        print(f"\n📊 Occupancy Rate Statistics:")
        print(f"  Mean: {rate_stats['mean']:.1f}%")
        print(f"  Median: {rate_stats['50%']:.1f}%")
        print(f"  Range: {rate_stats['min']:.1f}% to {rate_stats['max']:.1f}%")
        print(f"  Records with capacity data: {valid_capacity.sum():,} ({valid_capacity.sum()/len(adventhealth_clean)*100:.1f}%)")

        # Check for unrealistic occupancy rates (>100%)
        over_capacity = (adventhealth_clean['occupancy_rate'] > 100).sum()
        if over_capacity > 0:
            print(f"  ⚠️  Warning: {over_capacity} records ({over_capacity/len(adventhealth_clean)*100:.1f}%) show >100% occupancy")
            print(f"     This may indicate data quality issues or temporary capacity adjustments")

# Show sample of cleaned data (after quality filter)
print(f"\n📋 Sample of Cleaned Data (after quality filter, first 3 rows):")
sample_cols = ['collection_week', occupancy_col]
if capacity_col in adventhealth_clean.columns:
    sample_cols.append(capacity_col)
if 'occupancy_rate' in adventhealth_clean.columns:
    sample_cols.append('occupancy_rate')
if coverage_col in adventhealth_clean.columns:
    sample_cols.append(coverage_col)
display_cols = [col for col in sample_cols if col in adventhealth_clean.columns]
print(adventhealth_clean[display_cols].head(3).to_string(index=False))

# Extract additional metrics if available in the dataset
print(f"\n📊 Extracting additional metrics (ICU, ventilators, COVID patients, staffed beds)...")

# Define additional columns to extract if available
# Note: Column names may vary by dataset version - these are common patterns
additional_metrics = {
    'icu_occupied': 'staffed_adult_icu_bed_occupancy_7_day_avg',
    'icu_capacity': 'total_staffed_adult_icu_beds_7_day_avg',
    'ventilator_used': 'total_ventilators_in_use_7_day_avg',
    'ventilator_available': 'total_ventilators_available_7_day_avg',
    'covid_inpatients': 'total_adult_patients_hospitalized_confirmed_and_suspected_covid_7_day_avg',
    'covid_icu': 'staffed_icu_adult_patients_confirmed_covid_7_day_avg',
    'staffed_beds': 'all_adult_hospital_inpatient_beds_7_day_avg'  # Already have this as bed_capacity, but keep for consistency
}

# Try alternative column names if primary names don't exist
alternative_names = {
    'icu_occupied': [
        'staffed_adult_icu_bed_occupancy_7_day_avg',
        'staffed_icu_adult_patients_confirmed_and_suspected_covid_7_day_avg',
        'icu_bed_occupancy_7_day_avg'
    ],
    'icu_capacity': [
        'total_staffed_adult_icu_beds_7_day_avg',
        'total_icu_beds_7_day_avg',
        'icu_beds_7_day_avg'
    ],
    'ventilator_used': [
        'total_ventilators_in_use_7_day_avg',
        'ventilators_in_use_7_day_avg',
        'total_ventilators_7_day_avg'
    ],
    'ventilator_available': [
        'total_ventilators_available_7_day_avg',
        'ventilators_available_7_day_avg'
    ],
    'covid_inpatients': [
        'total_adult_patients_hospitalized_confirmed_and_suspected_covid_7_day_avg',
        'total_patients_hospitalized_covid_7_day_avg',
        'covid_patients_7_day_avg'
    ],
    'covid_icu': [
        'staffed_icu_adult_patients_confirmed_covid_7_day_avg',
        'total_patients_with_covid_19_in_icu_7_day_avg',
        'staffed_icu_adult_patients_confirmed_and_suspected_covid_7_day_avg',
        'covid_icu_patients_7_day_avg',
        'icu_covid_patients_7_day_avg'
    ],
    'staffed_beds': [
        'all_adult_hospital_inpatient_beds_7_day_avg',  # Already have this
        'total_staffed_beds_7_day_avg',
        'staffed_beds_7_day_avg'
    ]
}

# Add additional metrics to adventhealth_clean if they exist in the raw data
extracted_count = 0
for new_col, primary_raw_col in additional_metrics.items():
    # Skip if we already have this column (e.g., staffed_beds might be same as bed_capacity)
    if new_col in adventhealth_clean.columns:
        print(f"  ✓ '{new_col}' already exists (skipping)")
        extracted_count += 1
        continue

    # Try primary name first
    raw_col = None
    if primary_raw_col in adventhealth_df.columns:
        raw_col = primary_raw_col
    else:
        # Try alternative names
        for alt_name in alternative_names.get(new_col, []):
            if alt_name in adventhealth_df.columns:
                raw_col = alt_name
                break

    if raw_col:
        # Extract and clean the column
        adventhealth_clean[new_col] = adventhealth_df[raw_col].replace(missing_codes, np.nan)
        adventhealth_clean[new_col] = pd.to_numeric(adventhealth_clean[new_col], errors='coerce')

        # Count non-null values
        non_null_count = adventhealth_clean[new_col].notna().sum()
        print(f"  ✓ Extracted '{new_col}' from '{raw_col}' ({non_null_count}/{len(adventhealth_clean)} non-null values)")
        extracted_count += 1
    else:
        # Try to find similar column names (fuzzy matching)
        similar_cols = [col for col in adventhealth_df.columns
                       if new_col.split('_')[0] in col.lower() or
                       any(term in col.lower() for term in ['icu', 'ventilator', 'covid', 'staffed']
                           if term in new_col.lower())]
        if similar_cols:
            print(f"  ⚠️  '{new_col}': Primary column not found, but found similar columns: {similar_cols[:3]}")
        else:
            print(f"  ⚠️  '{new_col}': Column not found in dataset (will be skipped in KPIs)")

if extracted_count > 0:
    print(f"\n  ✓ Successfully extracted {extracted_count} additional metrics")
else:
    print(f"\n  ⚠️  No additional metrics extracted - dataset may not contain these columns")


🔍 Searching for hospital: ADVENTHEALTH ORLANDO
✓ Found: ADVENTHEALTH ORLANDO
  Location: ORLANDO, FL
  Records: 214 weekly records
  Date range: 2020-03-22 to 2024-04-21

📅 Collection Week Day of Week Analysis:
  Total collection dates: 214
    Sunday: 214 dates (100.0%)

  Most common day: Sunday (214 dates, 100.0%)

  Sample dates with day of week:
    2022-10-09 - Sunday
    2023-01-29 - Sunday
    2020-08-02 - Sunday
    2023-11-12 - Sunday
    2020-06-28 - Sunday

📋 Sample of Filtered Data (before cleaning, first 3 rows):
collection_week        hospital_name    city state all_adult_hospital_inpatient_bed_occupied_7_day_avg all_adult_hospital_inpatient_beds_7_day_avg  all_adult_hospital_inpatient_bed_occupied_7_day_coverage
     2022-10-09 ADVENTHEALTH ORLANDO ORLANDO    FL                                             2,363.1                                       2,460                                                         7
     2023-01-29 ADVENTHEALTH ORLANDO ORLANDO    FL      

## Create Bed Occupancy Time Series

The data is already weekly (7-day averages), so we'll create a time series with weekly frequency.

**Data Quality Approach:**
- Only weeks with valid occupancy data are included (quality data period: 2020-07-19 to 2024-04-21)
- Early weeks (2020-03-22 to 2020-07-12) are excluded due to missing occupancy data during early pandemic
- If any gaps exist within the quality period, they would be imputed using time-aware linear interpolation
- The `is_imputed` column marks which weeks were interpolated (0 for this dataset - no gaps within quality period)


In [21]:
# Create bed occupancy time series from weekly data
print(f"\n📅 Creating weekly time series from bed occupancy data...")

# Sort by collection week
adventhealth_clean = adventhealth_clean.sort_values('collection_week').reset_index(drop=True)

# Calculate date range from QUALITY DATA (adventhealth_clean), not from all records
# This ensures external data fetching uses only the date range for this specific hospital's actual data
date_range_start = adventhealth_clean['collection_week'].min()
date_range_end = adventhealth_clean['collection_week'].max()
full_range_start = adventhealth_df['collection_week'].min()
full_range_end = adventhealth_df['collection_week'].max()
quality_weeks = len(adventhealth_clean)
excluded_weeks = len(adventhealth_df) - quality_weeks

print(f"  Full date range (all records): {full_range_start.date()} to {full_range_end.date()}")
print(f"  Quality data date range: {date_range_start.date()} to {date_range_end.date()}")
print(f"  Quality weeks: {quality_weeks}, Excluded weeks: {excluded_weeks}")
if excluded_weeks > 0:
    print(f"  Note: Excluded weeks will be imputed to create complete time series")
else:
    print(f"  Note: All weeks have valid data - no imputation needed")

# Create weekly bed occupancy time series
bed_occupancy_weekly = pd.DataFrame({
    'date': adventhealth_clean['collection_week'],
    'bed_occupancy': adventhealth_clean[occupancy_col],
    'coverage': adventhealth_clean[coverage_col],
    'hospital_name': adventhealth_clean['hospital_name'],
    'city': adventhealth_clean['city'],
    'state': adventhealth_clean['state'],
})

# Show sample of time series data (before interpolation)
print(f"\n📋 Sample of Time Series Data (before interpolation, first 3 rows):")
print(bed_occupancy_weekly.head(3).to_string())

# Add capacity if available
if capacity_col in adventhealth_clean.columns:
    bed_occupancy_weekly['bed_capacity'] = adventhealth_clean[capacity_col]

# Add additional metrics if available
additional_metric_cols = ['icu_occupied', 'icu_capacity', 'ventilator_used', 'ventilator_available',
                          'covid_inpatients', 'covid_icu', 'staffed_beds']
for col in additional_metric_cols:
    if col in adventhealth_clean.columns:
        bed_occupancy_weekly[col] = adventhealth_clean[col]

# Calculate occupancy rate if capacity is available
if 'bed_capacity' in bed_occupancy_weekly.columns and bed_occupancy_weekly['bed_capacity'].notna().sum() > 0:
    bed_occupancy_weekly['occupancy_rate'] = (
        bed_occupancy_weekly['bed_occupancy'] / bed_occupancy_weekly['bed_capacity']
    ) * 100

# Calculate derived metrics for additional KPIs if available
# ICU occupancy rate
if 'icu_occupied' in bed_occupancy_weekly.columns and 'icu_capacity' in bed_occupancy_weekly.columns:
    mask = (bed_occupancy_weekly['icu_capacity'].notna()) & (bed_occupancy_weekly['icu_capacity'] > 0)
    if mask.sum() > 0:
        bed_occupancy_weekly.loc[mask, 'icu_occupancy_rate'] = (
            bed_occupancy_weekly.loc[mask, 'icu_occupied'] / bed_occupancy_weekly.loc[mask, 'icu_capacity']
        ) * 100

# Ventilator utilization rate
if 'ventilator_used' in bed_occupancy_weekly.columns and 'ventilator_available' in bed_occupancy_weekly.columns:
    mask = (bed_occupancy_weekly['ventilator_available'].notna()) & (bed_occupancy_weekly['ventilator_available'] > 0)
    if mask.sum() > 0:
        bed_occupancy_weekly.loc[mask, 'ventilator_utilization_rate'] = (
            bed_occupancy_weekly.loc[mask, 'ventilator_used'] / bed_occupancy_weekly.loc[mask, 'ventilator_available']
        ) * 100

# COVID patient percentage
if 'covid_inpatients' in bed_occupancy_weekly.columns and 'bed_occupancy' in bed_occupancy_weekly.columns:
    mask = (bed_occupancy_weekly['bed_occupancy'].notna()) & (bed_occupancy_weekly['bed_occupancy'] > 0)
    if mask.sum() > 0:
        bed_occupancy_weekly.loc[mask, 'covid_patient_pct'] = (
            bed_occupancy_weekly.loc[mask, 'covid_inpatients'] / bed_occupancy_weekly.loc[mask, 'bed_occupancy']
        ) * 100

# Set date as index for time series
bed_occupancy_weekly.set_index('date', inplace=True)

# Create complete weekly series (handle missing weeks AND excluded weeks)
print(f"\n🔍 Creating complete weekly series (including imputation of excluded weeks)...")

# Check if we have any data before proceeding
if len(bed_occupancy_weekly) == 0:
    raise ValueError("❌ No bed occupancy data available! Check data filtering steps.")

# Verify dates are valid
if bed_occupancy_weekly.index.isna().all():
    raise ValueError("❌ All dates are missing! Check date column in source data.")

# Use date range to include all weeks (including excluded ones)
# Ensure dates are properly aligned by using the actual data dates as reference
actual_start = bed_occupancy_weekly.index.min()
actual_end = bed_occupancy_weekly.index.max()

# Create weekly date range that matches the actual data range
# Collection dates are Sundays, so use W-SUN (or W which defaults to Sunday)
weekly_date_range = pd.date_range(
    start=date_range_start,
    end=date_range_end,
    freq='W-SUN'  # Week ending Sunday (matching COVID data collection_week which is all Sundays)
)

print(f"  Data date range: {actual_start.date()} to {actual_end.date()}")
print(f"  Target date range: {weekly_date_range.min().date()} to {weekly_date_range.max().date()}")
print(f"  Quality weeks in data: {len(bed_occupancy_weekly)}")
print(f"  Target weeks in range: {len(weekly_date_range)}")

# Reindex to complete weekly series (this will create NaNs for excluded weeks)
bed_occupancy_complete = bed_occupancy_weekly.reindex(weekly_date_range)

# Check if reindexing worked (if all values are NaN, dates don't align)
if bed_occupancy_complete['bed_occupancy'].isna().all() and len(bed_occupancy_weekly) > 0:
    print(f"  ⚠️  Warning: Date mismatch detected - dates in data don't match target range")
    print(f"     Data dates: {bed_occupancy_weekly.index.min().date()} to {bed_occupancy_weekly.index.max().date()}")
    print(f"     Target dates: {weekly_date_range.min().date()} to {weekly_date_range.max().date()}")
    print(f"     Using actual data dates instead of generated range")

    # Use actual dates from data instead (matching the day of week in the data)
    # Determine the day of week from the data
    first_date = bed_occupancy_weekly.index.min()
    day_name = first_date.strftime('%A')
    # Map day name to pandas frequency
    day_freq_map = {
        'Sunday': 'W-SUN',
        'Monday': 'W-MON',
        'Tuesday': 'W-TUE',
        'Wednesday': 'W-WED',
        'Thursday': 'W-THU',
        'Friday': 'W-FRI',
        'Saturday': 'W-SAT'
    }
    freq = day_freq_map.get(day_name, 'W-SUN')  # Default to Sunday

    actual_date_range = pd.date_range(
        start=bed_occupancy_weekly.index.min(),
        end=bed_occupancy_weekly.index.max(),
        freq=freq
    )
    bed_occupancy_complete = bed_occupancy_weekly.reindex(actual_date_range)
    weekly_date_range = actual_date_range  # Update for later use

# Count missing weeks before interpolation
missing_weeks_before = bed_occupancy_complete['bed_occupancy'].isna().sum()

# Verify we have some data after reindexing
if bed_occupancy_complete['bed_occupancy'].notna().sum() == 0:
    raise ValueError("❌ All bed occupancy values are missing after reindexing! Check date alignment.")

# Interpolate missing weeks using time-aware linear interpolation
# This is better than forward-fill as it creates smoother transitions
if missing_weeks_before > 0:
    print(f"  Found {missing_weeks_before} missing weeks ({missing_weeks_before/len(bed_occupancy_complete)*100:.1f}%)")
    print(f"    - Quality weeks with gaps: {max(0, missing_weeks_before - excluded_weeks)}")
    print(f"    - Excluded weeks (low coverage): {excluded_weeks}")
    print(f"  Interpolating all missing weeks using time-aware linear interpolation...")

    # Interpolate numeric columns (including additional metrics)
    numeric_cols = ['bed_occupancy', 'bed_capacity', 'occupancy_rate']
    # Add additional metrics to interpolation list if they exist
    additional_metric_cols = ['icu_occupied', 'icu_capacity', 'ventilator_used', 'ventilator_available',
                              'covid_inpatients', 'covid_icu', 'staffed_beds',
                              'icu_occupancy_rate', 'ventilator_utilization_rate', 'covid_patient_pct']
    for col in additional_metric_cols:
        if col in bed_occupancy_complete.columns:
            numeric_cols.append(col)

    for col in numeric_cols:
        if col in bed_occupancy_complete.columns:
            bed_occupancy_complete[col] = bed_occupancy_complete[col].interpolate(
                method='time',  # Time-aware interpolation (better than 'linear')
                limit_direction='both'  # Fill from both directions
            )

    # Mark imputed weeks for transparency
    bed_occupancy_complete['is_imputed'] = (
        ~bed_occupancy_complete.index.isin(bed_occupancy_weekly.index)
    ).astype(int)

    # Also add coverage information for imputed weeks (set to 0 to indicate low quality)
    if 'coverage' not in bed_occupancy_complete.columns:
        bed_occupancy_complete['coverage'] = None
    # For imputed weeks, set coverage to indicate they were excluded
    bed_occupancy_complete.loc[bed_occupancy_complete['is_imputed'] == 1, 'coverage'] = 0

    missing_weeks_after = bed_occupancy_complete['bed_occupancy'].isna().sum()
    if missing_weeks_after > 0:
        # Handle edge cases (beginning/end of series)
        # For beginning: Use linear extrapolation based on trend from first few quality weeks
        # For end: Use backward fill from last quality week

        # Find first and last quality data points
        first_quality_idx = bed_occupancy_complete['bed_occupancy'].first_valid_index()
        last_quality_idx = bed_occupancy_complete['bed_occupancy'].last_valid_index()

        # Handle beginning of series (before first quality data)
        if first_quality_idx is not None:
            first_quality_date = bed_occupancy_complete.index.get_loc(first_quality_idx)
            if first_quality_date > 0:
                # For beginning of series, use forward fill from first quality data point
                # This is simpler and more reliable than extrapolation when all initial weeks are missing
                print(f"  Using forward fill for {first_quality_date} weeks at beginning of series...")
                for col in numeric_cols:
                    if col in bed_occupancy_complete.columns:
                        bed_occupancy_complete[col] = bed_occupancy_complete[col].ffill()
                print(f"    ✓ Forward-filled {first_quality_date} weeks at beginning from first quality data point")

        # Handle end of series (after last quality data)
        if last_quality_idx is not None:
            last_quality_date = bed_occupancy_complete.index.get_loc(last_quality_idx)
            if last_quality_date < len(bed_occupancy_complete) - 1:
                # Use backward fill for end of series
                for col in numeric_cols:
                    if col in bed_occupancy_complete.columns and bed_occupancy_complete[col].isna().sum() > 0:
                        bed_occupancy_complete[col] = bed_occupancy_complete[col].bfill()

        # Final check: fill any remaining NaNs with forward/backward fill
        for col in numeric_cols:
            if col in bed_occupancy_complete.columns and bed_occupancy_complete[col].isna().sum() > 0:
                bed_occupancy_complete[col] = bed_occupancy_complete[col].ffill().bfill()

        missing_weeks_final = bed_occupancy_complete['bed_occupancy'].isna().sum()
        print(f"  ✓ Interpolated {missing_weeks_before - missing_weeks_after} weeks")
        if missing_weeks_final > 0:
            print(f"  ⚠️  {missing_weeks_final} weeks still missing after all interpolation methods")
        else:
            print(f"  ✓ All missing weeks filled (interpolation + extrapolation/fill)")
    else:
        print(f"  ✓ Successfully interpolated all {missing_weeks_before} missing weeks")

    # Show sample of interpolated data if any imputation occurred
    if missing_weeks_before > 0:
        imputed_mask = bed_occupancy_complete['is_imputed'] == 1
        if imputed_mask.sum() > 0:
            print(f"\n📋 Sample of Interpolated Weeks (showing {min(3, imputed_mask.sum())} imputed rows):")
            imputed_sample = bed_occupancy_complete[imputed_mask].head(3)
            sample_cols = ['bed_occupancy']
            if 'bed_capacity' in imputed_sample.columns:
                sample_cols.append('bed_capacity')
            if 'occupancy_rate' in imputed_sample.columns:
                sample_cols.append('occupancy_rate')
            sample_cols.append('is_imputed')
            display_cols = [col for col in sample_cols if col in imputed_sample.columns]
            print(imputed_sample[display_cols].to_string())
else:
    bed_occupancy_complete['is_imputed'] = 0
    print(f"  ✓ No missing weeks found - complete weekly series")
    # Ensure coverage column exists
    if 'coverage' not in bed_occupancy_complete.columns:
        bed_occupancy_complete['coverage'] = 7  # Full coverage for quality weeks

# For compatibility with existing pipeline, create columns matching expected format
bed_occupancy_complete['aligned_date'] = bed_occupancy_complete.index

# Create final cleaned dataset (weekly, not daily)
occupancy_clean = bed_occupancy_complete.reset_index().rename(columns={'index': 'date'})

# Remove any unexpected columns that shouldn't be in the output
# Only keep expected columns for bed occupancy data (including additional metrics if available)
expected_cols = ['date', 'aligned_date', 'bed_occupancy', 'bed_capacity', 'occupancy_rate',
                 'coverage', 'hospital_name', 'city', 'state', 'is_imputed']
# Add additional metrics to expected columns if they exist
additional_metric_cols = ['icu_occupied', 'icu_capacity', 'ventilator_used', 'ventilator_available',
                          'covid_inpatients', 'covid_icu', 'staffed_beds',
                          'icu_occupancy_rate', 'ventilator_utilization_rate', 'covid_patient_pct']
for col in additional_metric_cols:
    if col in occupancy_clean.columns:
        expected_cols.append(col)
occupancy_clean = occupancy_clean[[col for col in expected_cols if col in occupancy_clean.columns]]

# Remove any records with missing bed occupancy (should be none after interpolation)
initial_count = len(occupancy_clean)
occupancy_clean = occupancy_clean.dropna(subset=['bed_occupancy', 'aligned_date'])
dropped_count = initial_count - len(occupancy_clean)

# Validate that no records were unexpectedly dropped
if dropped_count > 0:
    print(f"  ⚠️  Warning: Dropped {dropped_count} records with missing bed_occupancy or aligned_date")
    print(f"     This should not happen after interpolation - please investigate")
    if dropped_count == initial_count:
        print(f"  ❌ CRITICAL: All records were dropped! This means interpolation failed.")
        print(f"     Possible causes:")
        print(f"     1. Date mismatch between data dates and target date range")
        print(f"     2. All values were NaN after reindexing")
        print(f"     3. Interpolation couldn't fill missing values")
        raise ValueError("❌ All records dropped - interpolation failed. Check date alignment and data quality.")
else:
    print(f"  ✓ All records have valid bed_occupancy and aligned_date (no records dropped)")

# Sort by date
occupancy_clean = occupancy_clean.sort_values('aligned_date').reset_index(drop=True)

# Validate date consistency
validate_dataframe(occupancy_clean, "Date consistency", date_col='aligned_date')

print(f"\n✓ Weekly bed occupancy time series created:")
print(f"  Total weekly records: {len(occupancy_clean):,}")
print(f"  Date range: {occupancy_clean['aligned_date'].min().date()} to {occupancy_clean['aligned_date'].max().date()}")
print(f"  Data frequency: Weekly (7-day averages)")
if 'is_imputed' in occupancy_clean.columns:
    imputed_count = occupancy_clean['is_imputed'].sum()
    quality_count = len(occupancy_clean) - imputed_count
    if imputed_count > 0:
        print(f"  Quality weeks: {quality_count} ({quality_count/len(occupancy_clean)*100:.1f}%)")
        print(f"  Imputed weeks: {imputed_count} ({imputed_count/len(occupancy_clean)*100:.1f}%)")
        print(f"    Note: Imputed weeks include excluded weeks (low coverage) that were interpolated")
    else:
        print(f"  Imputed weeks: 0 (complete data)")

if len(occupancy_clean) > 0:
    print(f"\n📊 Bed Occupancy Summary (Weekly):")
    print(f"  Mean: {occupancy_clean['bed_occupancy'].mean():.1f} beds")
    print(f"  Median: {occupancy_clean['bed_occupancy'].median():.1f} beds")
    print(f"  Min: {occupancy_clean['bed_occupancy'].min():.1f} beds")
    print(f"  Max: {occupancy_clean['bed_occupancy'].max():.1f} beds")

    # Outlier detection: Check for unrealistic values
    occupancy = occupancy_clean['bed_occupancy']
    negative_count = (occupancy < 0).sum()
    if negative_count > 0:
        print(f"  ⚠️  Warning: Found {negative_count} records with negative occupancy (unrealistic)")

    # Check for extremely high values (more than 3 standard deviations above mean)
    if len(occupancy) > 10:  # Need enough data for meaningful stats
        mean_occ = occupancy.mean()
        std_occ = occupancy.std()
        outlier_threshold = mean_occ + 3 * std_occ
        outliers = (occupancy > outlier_threshold).sum()
        if outliers > 0:
            print(f"  ⚠️  Warning: Found {outliers} potential outliers (>3σ above mean: {outlier_threshold:.1f} beds)")

    # Validate data types
    if not pd.api.types.is_numeric_dtype(occupancy_clean['bed_occupancy']):
        print(f"  ⚠️  Warning: bed_occupancy is not numeric type: {occupancy_clean['bed_occupancy'].dtype}")
    else:
        print(f"  ✓ Data type validated: bed_occupancy is numeric ({occupancy_clean['bed_occupancy'].dtype})")

    if 'occupancy_rate' in occupancy_clean.columns:
        print(f"\n📊 Occupancy Rate Summary (Weekly):")
        print(f"  Mean: {occupancy_clean['occupancy_rate'].mean():.1f}%")
        print(f"  Median: {occupancy_clean['occupancy_rate'].median():.1f}%")
        print(f"  Range: {occupancy_clean['occupancy_rate'].min():.1f}% to {occupancy_clean['occupancy_rate'].max():.1f}%")

# Show sample of final cleaned data
print(f"\n📋 Sample of Final Cleaned Data (first 5 rows):")
sample_cols = ['date', 'aligned_date', 'bed_occupancy']
if 'bed_capacity' in occupancy_clean.columns:
    sample_cols.append('bed_capacity')
if 'occupancy_rate' in occupancy_clean.columns:
    sample_cols.append('occupancy_rate')
if 'is_imputed' in occupancy_clean.columns:
    sample_cols.append('is_imputed')
display_cols = [col for col in sample_cols if col in occupancy_clean.columns]
print(occupancy_clean[display_cols].head(5).to_string(index=False))
print(f"\n📋 Sample of Final Cleaned Data (last 5 rows):")
print(occupancy_clean[display_cols].tail(5).to_string(index=False))

# ============================================================================
# DEFINE HOSPITAL DATE RANGE FOR EXTERNAL DATA EXTRACTION
# ============================================================================
# These variables will be used by the weather and flu data extraction cells
# to fetch data ONLY for the date range that this specific hospital has data

HOSPITAL_START_DATE = pd.to_datetime(occupancy_clean['aligned_date'].min())
HOSPITAL_END_DATE = pd.to_datetime(occupancy_clean['aligned_date'].max())

# Derive year range from hospital data
HOSPITAL_START_YEAR = HOSPITAL_START_DATE.year
HOSPITAL_END_YEAR = HOSPITAL_END_DATE.year

print(f"\n" + "="*60)
print(f"HOSPITAL DATE RANGE (for external data extraction)")
print(f"="*60)
print(f"  Hospital: {SELECTED_HOSPITAL}")
print(f"  Start date: {HOSPITAL_START_DATE.date()} ({HOSPITAL_START_DATE.strftime('%A')})")
print(f"  End date: {HOSPITAL_END_DATE.date()} ({HOSPITAL_END_DATE.strftime('%A')})")
print(f"  Year range: {HOSPITAL_START_YEAR} to {HOSPITAL_END_YEAR}")
print(f"  Total weeks: {len(occupancy_clean)}")
print(f"\n  ℹ️  External data (weather, flu) will be fetched for this specific date range")


📅 Creating weekly time series from bed occupancy data...
  Full date range (all records): 2020-03-22 to 2024-04-21
  Quality data date range: 2020-07-19 to 2024-04-21
  Quality weeks: 197, Excluded weeks: 17
  Note: Excluded weeks will be imputed to create complete time series

📋 Sample of Time Series Data (before interpolation, first 3 rows):
        date  bed_occupancy  coverage         hospital_name     city state
0 2020-07-19         2011.0         3  ADVENTHEALTH ORLANDO  ORLANDO    FL
1 2020-07-26         2022.9         7  ADVENTHEALTH ORLANDO  ORLANDO    FL
2 2020-08-02         2007.7         7  ADVENTHEALTH ORLANDO  ORLANDO    FL

🔍 Creating complete weekly series (including imputation of excluded weeks)...
  Data date range: 2020-07-19 to 2024-04-21
  Target date range: 2020-07-19 to 2024-04-21
  Quality weeks in data: 197
  Target weeks in range: 197
  ✓ No missing weeks found - complete weekly series
  ✓ All records have valid bed_occupancy and aligned_date (no records drop

## Date Verification

For COVID-19 dataset, dates are real (not deidentified), so no date alignment is needed. The `aligned_date` column represents the collection week date (no transformation needed).


In [22]:
# Verify date columns for COVID data (no alignment needed - dates are real)
if len(occupancy_clean) > 0:
    print(f"\n✓ Date Verification (COVID data uses real dates):")
    print(f"  Total records: {len(occupancy_clean):,}")
    print(f"  Date range: {occupancy_clean['aligned_date'].min().date()} to {occupancy_clean['aligned_date'].max().date()}")

    # Verify aligned_date column exists (as expected for COVID data)
    if 'aligned_date' in occupancy_clean.columns:
        print(f"  ✓ aligned_date column present (no transformation needed)")
    else:
        print(f"  ⚠️  Warning: aligned_date column missing")

    print(f"  Data frequency: Weekly (7-day averages)")
else:
    print("⚠️  No cleaned bed occupancy data available")


✓ Date Verification (COVID data uses real dates):
  Total records: 197
  Date range: 2020-07-19 to 2024-04-21
  ✓ aligned_date column present (no transformation needed)
  Data frequency: Weekly (7-day averages)


# Part 2: External Data Extraction

Extract external data sources (weather and flu) for Orlando, FL (2020-2024) to enhance forecasting models. These are **optional** but recommended.

**Note:**
- Weather data will be downloaded for Orlando, FL coordinates
- Flu data from CDC FluView API is US-specific and available for Florida
- These external features can help improve bed occupancy forecasting accuracy


### 2.1 Install Required Packages

Install packages needed for fetching weather and flu data. These are only required for external data extraction.


In [23]:
# Install required packages for external data extraction
# Meteostat: For weather data (uses NOAA sources)
# requests: For CDC FluView API (usually pre-installed)

ensure_package('requests')
ensure_package('meteostat')
from meteostat import Point, Daily

print("\n✅ All required packages ready for external data extraction")

✓ requests already available
✓ meteostat already available

✅ All required packages ready for external data extraction


### Weather Data Extraction

In [24]:
# ============================================================
# WEATHER DATA EXTRACTION
# ============================================================

print("\n" + "="*60)
print("WEATHER DATA EXTRACTION (Orlando, FL)")
print("="*60)

# Years needed - derived from hospital's actual date range
# This ensures we fetch weather data only for the years the hospital has data
weather_years = list(range(HOSPITAL_START_YEAR, HOSPITAL_END_YEAR + 1))
print(f"  Using hospital date range: {HOSPITAL_START_DATE.date()} to {HOSPITAL_END_DATE.date()}")
print(f"  Weather years to fetch: {weather_years}")

# Orlando coordinates (28.5383°N, 81.3792°W)
orlando_coords = (28.5383, -81.3792)

# Check if weather data already exists
weather_file_local = os.path.join(DATA_EXTERNAL, 'weather_orlando.csv')
weather_file_drive = os.path.join(DRIVE_DATA_EXTERNAL, 'weather_orlando.csv') if DRIVE_DATA_EXTERNAL else None

weather_df = None

# Try loading existing file
for weather_file in [weather_file_drive, weather_file_local]:
    if weather_file and os.path.exists(weather_file):
        try:
            weather_df = pd.read_csv(weather_file, parse_dates=['date'])
            print(f"✓ Found existing weather data: {weather_file}")
            print(f"  Records: {len(weather_df):,}")

            # Check if file has actual data (non-null values)
            numeric_cols_check = weather_df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols_check) > 0:
                non_null_count = weather_df[numeric_cols_check].notna().sum().sum()
                if non_null_count == 0:
                    print(f"  ⚠️  WARNING: Weather file exists but has NO data (all NaN values)")
                    print(f"     File will be ignored and data will be re-downloaded")
                    weather_df = None  # Reset to trigger re-download
                    continue

            if len(weather_df) > 0:
                print(f"  Date range: {weather_df['date'].min().date()} to {weather_df['date'].max().date()}")

                # Check if data is daily or weekly (weekly should have ~214 records for 2020-2024)
                # If it has >500 records, it's likely daily and needs aggregation
                if len(weather_df) > 500:
                    print(f"  ⚠️  Detected daily data - will aggregate to weekly")
                else:
                    print(f"  ✓ Data appears to be weekly (already aggregated)")
                break
            else:
                print(f"  ⚠️  WARNING: Weather file is empty (0 records)")
                print(f"     File will be ignored and data will be re-downloaded")
                weather_df = None  # Reset to trigger re-download
                continue
        except Exception as e:
            print(f"⚠️  Error loading {weather_file}: {e}")
            weather_df = None  # Reset to trigger re-download

# If not found, download using Meteostat
if weather_df is None:
    print("\n📥 Downloading weather data using Meteostat...")
    print(f"  Location: Orlando, FL ({orlando_coords[0]}°N, {abs(orlando_coords[1])}°W)")
    print(f"  Years: {weather_years[0]} to {weather_years[-1]} ({len(weather_years)} years)")

    try:
        from meteostat import Point, Daily
        from datetime import datetime

        # Orlando coordinates (28.5383°N, 81.3792°W)
        orlando = Point(orlando_coords[0], orlando_coords[1])

        weather_data = []

        for year in weather_years:
            try:
                print(f"    Downloading {year}...", end=" ")
                data = Daily(orlando, datetime(year, 1, 1), datetime(year, 12, 31))
                data = data.fetch()

                if len(data) > 0:
                    data.reset_index(inplace=True)
                    data.rename(columns={'time': 'date'}, inplace=True)
                    weather_data.append(data)
                    print(f"✓ {len(data)} days")
                else:
                    print(f"⚠️  No data")
            except Exception as e:
                print(f"⚠️  Error: {e}")

        if weather_data:
            weather_df = pd.concat(weather_data, ignore_index=True)

            # Select and rename columns
            column_mapping = {
                'tavg': 'temp_avg',
                'tmax': 'temp_max',
                'tmin': 'temp_min',
                'prcp': 'precipitation',
                'wspd': 'wind_speed',
                'pres': 'pressure',
            }

            # Keep only available columns
            available_cols = ['date'] + [col for col in column_mapping.keys() if col in weather_df.columns]
            weather_df = weather_df[available_cols]
            weather_df.rename(columns=column_mapping, inplace=True)

            # Convert temperature from Celsius to Fahrenheit
            for temp_col in ['temp_avg', 'temp_max', 'temp_min']:
                if temp_col in weather_df.columns:
                    weather_df[temp_col] = weather_df[temp_col] * 9/5 + 32

            # Convert precipitation from mm to inches
            if 'precipitation' in weather_df.columns:
                weather_df['precipitation'] = weather_df['precipitation'] * 0.0393701

            # Sort by date
            weather_df['date'] = pd.to_datetime(weather_df['date'])
            weather_df = weather_df.sort_values('date').reset_index(drop=True)

            print(f"\n✓ Weather data downloaded: {len(weather_df):,} days")
        else:
            print("\n⚠️  No weather data downloaded")

    except ImportError:
        print("⚠️  Meteostat library not installed.")
        print("   Install with: pip install meteostat")
        print("   Or download manually from: https://www.ncei.noaa.gov/")
    except Exception as e:
        print(f"⚠️  Error downloading weather data: {e}")

# Clean and validate weather data
if weather_df is not None and len(weather_df) > 0:
    print("\n🧹 Cleaning weather data...")

    initial_count = len(weather_df)

    # Remove duplicates
    weather_df = weather_df.drop_duplicates(subset=['date']).reset_index(drop=True)

    # Ensure date column is datetime
    weather_df['date'] = pd.to_datetime(weather_df['date'])

    # Remove invalid dates
    weather_df = weather_df[weather_df['date'].notna()]

    # Sort by date
    weather_df = weather_df.sort_values('date').reset_index(drop=True)

    # Fill missing values in numeric columns (forward-fill, then backward-fill)
    numeric_cols = weather_df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col != 'date':
            weather_df[col] = weather_df[col].ffill().bfill()

    # Calculate temp_avg if missing
    if 'temp_avg' not in weather_df.columns or weather_df['temp_avg'].isna().all():
        if 'temp_max' in weather_df.columns and 'temp_min' in weather_df.columns:
            weather_df['temp_avg'] = (weather_df['temp_max'] + weather_df['temp_min']) / 2.0

    # Validate required columns
    required_cols = ['date', 'temp_avg']
    missing_cols = [col for col in required_cols if col not in weather_df.columns]

    if missing_cols:
        print(f"⚠️  Missing required columns: {missing_cols}")
        weather_df = None
    else:
        daily_count = len(weather_df)
        print(f"✓ Weather data cleaned: {daily_count:,} records (removed {initial_count - daily_count} duplicates/invalid)")
        print(f"  Date range: {weather_df['date'].min().date()} to {weather_df['date'].max().date()}")
        print(f"  Columns: {', '.join(weather_df.columns)}")

        # Check if data needs aggregation (if >500 records, it's likely daily)
        needs_aggregation = daily_count > 500

        if needs_aggregation:
            # Aggregate daily weather to weekly to match bed occupancy frequency
            print(f"\n📊 Aggregating weather data to weekly (matching bed occupancy frequency)...")
            weather_df['date'] = pd.to_datetime(weather_df['date'])
            weather_df_indexed = weather_df.set_index('date')

            # Build aggregation dictionary - only aggregate columns that exist
            agg_dict = {}
            if 'temp_avg' in weather_df_indexed.columns:
                agg_dict['temp_avg'] = 'mean'
            if 'temp_max' in weather_df_indexed.columns:
                agg_dict['temp_max'] = 'mean'
            if 'temp_min' in weather_df_indexed.columns:
                agg_dict['temp_min'] = 'mean'
            if 'precipitation' in weather_df_indexed.columns:
                agg_dict['precipitation'] = 'sum'  # Sum precipitation for the week
            if 'wind_speed' in weather_df_indexed.columns:
                agg_dict['wind_speed'] = 'mean'
            if 'pressure' in weather_df_indexed.columns:
                agg_dict['pressure'] = 'mean'

            # Resample to weekly (week ending on Sunday, matching hospital data collection_week)
            # COVID hospital data uses collection_week which are all Sundays
            weather_weekly = weather_df_indexed.resample('W-SUN').agg(agg_dict)

            # Reset index
            weather_weekly = weather_weekly.reset_index()

            print(f"  Daily records: {daily_count:,}")
            print(f"  Weekly records: {len(weather_weekly):,}")
            print(f"  Date range: {weather_weekly['date'].min().date()} to {weather_weekly['date'].max().date()}")

            # DIAGNOSTIC: Verify day of week after aggregation
            first_weather_date = weather_weekly['date'].iloc[0]
            last_weather_date = weather_weekly['date'].iloc[-1]
            print(f"\n  📊 WEATHER AGGREGATION DIAGNOSTICS:")
            print(f"     Aggregation frequency: W-SUN (week ending Sunday)")
            print(f"     First date: {first_weather_date.date()} ({first_weather_date.strftime('%A')})")
            print(f"     Last date: {last_weather_date.date()} ({last_weather_date.strftime('%A')})")
            if first_weather_date.strftime('%A') != 'Sunday':
                print(f"     ⚠️  WARNING: Weather dates are NOT Sundays! May cause alignment issues.")
            else:
                print(f"     ✓ Weather dates are Sundays (matches hospital collection_week)")

            # Use weekly data going forward
            weather_df = weather_weekly.copy()
        else:
            print(f"\n✓ Weather data is already weekly (no aggregation needed)")

        # Ensure weather data covers full hospital date range
        print(f"\n📅 Aligning weather data to hospital date range...")

        # Get hospital dates for alignment - try multiple sources
        hospital_weekly_dates = None

        # First try: occupancy_clean (final cleaned data)
        if 'occupancy_clean' in globals() and len(occupancy_clean) > 0 and 'aligned_date' in occupancy_clean.columns:
            hospital_start = occupancy_clean['aligned_date'].min()
            hospital_end = occupancy_clean['aligned_date'].max()

            if not pd.isna(hospital_start) and not pd.isna(hospital_end):
                print(f"  Hospital date range (from occupancy_clean): {hospital_start.date()} to {hospital_end.date()}")
                hospital_weekly_dates = pd.to_datetime(occupancy_clean['aligned_date']).unique()
                hospital_weekly_dates = pd.DatetimeIndex(sorted(hospital_weekly_dates))

        # Fallback: Use date range from raw data if occupancy_clean not available or empty
        if (hospital_weekly_dates is None or len(hospital_weekly_dates) == 0) and 'adventhealth_df' in globals():
            if 'collection_week' in adventhealth_df.columns:
                hospital_start = adventhealth_df['collection_week'].min()
                hospital_end = adventhealth_df['collection_week'].max()

                if not pd.isna(hospital_start) and not pd.isna(hospital_end):
                    print(f"  Hospital date range (from raw data): {hospital_start.date()} to {hospital_end.date()}")
                    # Create weekly date range matching hospital data frequency
                    # Hospital collection dates are Sundays, so use W-SUN
                    hospital_weekly_dates = pd.date_range(
                        start=hospital_start,
                        end=hospital_end,
                        freq='W-SUN'  # Week ending Sunday (matching COVID data collection_week)
                    )

        weather_indexed = weather_df.set_index('date')
        weather_min = weather_indexed.index.min()
        weather_max = weather_indexed.index.max()
        print(f"  Weather data date range (before alignment): {weather_min.date() if not pd.isna(weather_min) else 'NaT'} to {weather_max.date() if not pd.isna(weather_max) else 'NaT'}")
        print(f"  Weather records (before alignment): {len(weather_indexed)}")

        # Align weather to hospital dates if available
        if hospital_weekly_dates is not None and len(hospital_weekly_dates) > 0:
            # Check for date overlap (within 3 days tolerance for weekly data)
            weather_dates = weather_indexed.index
            overlap_count = sum(1 for h_date in hospital_weekly_dates
                              if (abs(weather_dates - h_date).min().days <= 3))
            print(f"  Date overlap: {overlap_count}/{len(hospital_weekly_dates)} hospital dates have matching weather data (within 3 days)")

            # Check if weather data has any non-null values
            numeric_cols_before = weather_indexed.select_dtypes(include=[np.number]).columns
            if len(numeric_cols_before) > 0:
                non_null_count = weather_indexed[numeric_cols_before].notna().sum().sum()
                print(f"  Weather data non-null values (before alignment): {non_null_count}")
                if non_null_count == 0:
                    print(f"  ⚠️  WARNING: Weather file has no actual data! All values are NaN.")
                    print(f"     Solution: Delete the weather file and re-download using Meteostat.")

            # Align weather data to exact hospital dates
            weather_aligned = align_data_to_dates(weather_df, hospital_weekly_dates, tolerance_days=3)

            if len(weather_aligned) == 0:
                print(f"  ⚠️  Warning: Alignment failed - using weather data as-is")
                weather_aligned = weather_indexed.copy()
            else:
                # Ensure dates match exactly (reindex to hospital dates to fill any gaps)
                weather_aligned = weather_aligned.reindex(hospital_weekly_dates)
                print(f"  ✓ Weather data aligned to {len(hospital_weekly_dates)} hospital dates")

                # Check if we have any data after alignment
                numeric_cols_check = weather_aligned.select_dtypes(include=[np.number]).columns
                has_data_after = False
                if len(numeric_cols_check) > 0:
                    has_data_after = weather_aligned[numeric_cols_check].notna().sum().sum() > 0

                if not has_data_after:
                    print(f"  ⚠️  Warning: No weather data after alignment - this may indicate date mismatch")
                    print(f"     Using original weather data without alignment")
                    weather_aligned = weather_indexed.copy()
        else:
            print(f"  ⚠️  Warning: Hospital dates not available - weather data not aligned")
            print(f"     Weather data will be saved as-is (can be aligned later)")
            weather_aligned = weather_indexed.copy()

        # Fill missing weeks with interpolation (for gaps in the middle)
        # Only interpolate if we have some non-null values to work with
        if len(weather_aligned) > 0:
            numeric_cols = weather_aligned.select_dtypes(include=[np.number]).columns
            has_data = False
            for col in numeric_cols:
                if weather_aligned[col].notna().sum() > 0:
                    has_data = True
                    break

            if has_data:
                # We have some data, so interpolation can work
                for col in numeric_cols:
                    weather_aligned[col] = weather_aligned[col].interpolate(method='time', limit_direction='both')

                # Forward/backward fill for edges if needed
                for col in numeric_cols:
                    weather_aligned[col] = weather_aligned[col].ffill().bfill()
                print(f"  ✓ Interpolated missing values in weather data")
            else:
                # No data to interpolate from - all values are NaN
                print(f"  ⚠️  WARNING: Cannot interpolate - weather data has no non-null values!")
                print(f"     This may indicate a date mismatch or missing weather data.")
                print(f"     Weather data will be saved as-is (may need manual alignment later)")

        # Reset index and rename
        weather_df = weather_aligned.reset_index().rename(columns={'index': 'date'})

        # Validate alignment - ensure dates match exactly
        if hospital_weekly_dates is not None and len(hospital_weekly_dates) > 0:
            weather_dates = pd.to_datetime(weather_df['date']).unique()
            hospital_dates_set = set(hospital_weekly_dates)
            weather_dates_set = set(weather_dates)

            if len(weather_df) == len(hospital_weekly_dates) and weather_dates_set == hospital_dates_set:
                print(f"  ✓ Weather records: {len(weather_df)} (exactly aligned to {len(hospital_weekly_dates)} hospital weeks)")
                print(f"  ✓ Date match: All weather dates match hospital dates exactly")
            else:
                missing_dates = hospital_dates_set - weather_dates_set
                extra_dates = weather_dates_set - hospital_dates_set
                if missing_dates:
                    print(f"  ⚠️  Warning: {len(missing_dates)} hospital dates missing from weather data")
                if extra_dates:
                    print(f"  ⚠️  Warning: {len(extra_dates)} weather dates not in hospital dates")

            weather_min = weather_df['date'].min()
            weather_max = weather_df['date'].max()
            print(f"  Weather date range: {weather_min.date() if not pd.isna(weather_min) else 'NaT'} to {weather_max.date() if not pd.isna(weather_max) else 'NaT'}")
            print(f"  Hospital date range: {hospital_weekly_dates.min().date()} to {hospital_weekly_dates.max().date()}")

            # Check for any remaining missing values
            missing_weather = weather_df.isnull().sum().sum()
            if missing_weather > 0:
                print(f"  ⚠️  Warning: {missing_weather} missing values in weather data after alignment")
            else:
                print(f"  ✓ Weather data now fully covers hospital date range (no missing values)")
        else:
            weather_min = weather_df['date'].min()
            weather_max = weather_df['date'].max()
            print(f"  Weather date range: {weather_min.date() if not pd.isna(weather_min) else 'NaT'} to {weather_max.date() if not pd.isna(weather_max) else 'NaT'}")
            print(f"  ⚠️  Note: Weather data not aligned to hospital dates (hospital dates unavailable)")

        # Save cleaned weather data (save to both locations)
        save_to_locations(weather_df, 'weather_orlando.csv', DATA_EXTERNAL, DRIVE_DATA_EXTERNAL)
else:
    print("\n⚠️  No weather data available")
    print("   You can:")
    print("   1. Install meteostat: pip install meteostat")
    print("   2. Download manually from: https://www.ncei.noaa.gov/")
    print("   3. Save as: data/external/weather_orlando.csv")


WEATHER DATA EXTRACTION (Orlando, FL)
  Using hospital date range: 2020-07-19 to 2024-04-21
  Weather years to fetch: [2020, 2021, 2022, 2023, 2024]
✓ Found existing weather data: /content/drive/MyDrive/hospital_occupancy_forecasting/data/external/weather_orlando.csv
  Records: 197
  Date range: 2020-07-19 to 2024-04-21
  ✓ Data appears to be weekly (already aggregated)

🧹 Cleaning weather data...
✓ Weather data cleaned: 197 records (removed 0 duplicates/invalid)
  Date range: 2020-07-19 to 2024-04-21
  Columns: date, temp_avg, temp_max, temp_min, precipitation, wind_speed, pressure

✓ Weather data is already weekly (no aggregation needed)

📅 Aligning weather data to hospital date range...
  Hospital date range (from occupancy_clean): 2020-07-19 to 2024-04-21
  Weather data date range (before alignment): 2020-07-19 to 2024-04-21
  Weather records (before alignment): 197
  Date overlap: 197/197 hospital dates have matching weather data (within 3 days)
  Weather data non-null values (be

### Flu Data Extraction

#### Option: Use Seasonal Indicators as Proxy

If CDC flu data is unavailable, we can create a seasonal proxy based on date patterns (higher activity in winter/spring, lower in summer/fall). This is a fallback option.


In [25]:
# Alternative: Create flu-like activity from date-based seasonal patterns
# This creates a seasonal proxy based on month patterns (not from dataset column)
# Winter and Spring typically have higher flu activity

# Initialize flu_df if not already defined (this cell runs before the main flu extraction)
if 'flu_df' not in globals():
    flu_df = None

if flu_df is None or (hasattr(flu_df, '__len__') and len(flu_df) == 0):
    print("\n📊 Creating flu activity proxy from date-based seasonal patterns...")

    # Create seasonal proxy based on month (if we have occupancy_clean with dates)
    if len(occupancy_clean) > 0 and 'aligned_date' in occupancy_clean.columns:
        # Create flu dataframe from weekly date patterns (matching bed occupancy frequency)
        flu_from_seasonal = pd.DataFrame({
            'date': occupancy_clean['aligned_date']
        })
        flu_from_seasonal['date'] = pd.to_datetime(flu_from_seasonal['date'])
        flu_from_seasonal['flu_activity_level'] = flu_from_seasonal['date'].apply(get_seasonal_flu_activity)

        # Remove duplicates (in case of multiple records per week)
        flu_df = flu_from_seasonal.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)

        print(f"✓ Created flu activity proxy from date-based seasonal patterns")
        print(f"  Records: {len(flu_df):,} (weekly frequency, matching bed occupancy data)")
        print(f"  Date range: {flu_df['date'].min().date()} to {flu_df['date'].max().date()}")
        print(f"  Activity levels: {flu_df['flu_activity_level'].min():.2f} to {flu_df['flu_activity_level'].max():.2f}")
        print(f"  Note: This is a proxy based on seasonality patterns, not actual flu data")
    else:
        print("  ⚠️  No date data available for seasonal proxy")
        print("     Creating empty flu dataframe")
        flu_df = pd.DataFrame(columns=['date', 'flu_activity_level'])

In [26]:
# ============================================================
# FLU DATA EXTRACTION
# ============================================================

print("\n" + "="*60)
print("FLU DATA EXTRACTION (Florida, US)")
print("="*60)
print("\n📋 Options for Florida flu data:")
print("   1. CDC FluView API (US data, includes Florida)")
print("   2. Use seasonal indicators as proxy (if CDC unavailable)")
print("   3. Skip flu data (models work without it)")
print("\n✅ Note: CDC FluView is perfect for US hospitals like ADVENTHEALTH ORLANDO in Florida.")

# Years needed - derived from hospital's actual date range
# This ensures we fetch flu data only for the years the hospital has data
flu_years = list(range(HOSPITAL_START_YEAR, HOSPITAL_END_YEAR + 1))
print(f"  Using hospital date range: {HOSPITAL_START_DATE.date()} to {HOSPITAL_END_DATE.date()}")
print(f"  Flu years to fetch: {flu_years}")

# File paths for flu data
flu_file_local = os.path.join(DATA_EXTERNAL, 'flu_cdc.csv')
flu_file_drive = os.path.join(DRIVE_DATA_EXTERNAL, 'flu_cdc.csv') if DRIVE_DATA_EXTERNAL else None

# Check if flu data already exists (try multiple sources)
flu_file_paths = [
    ('CDC FluView', flu_file_local),
    ('CDC FluView (Drive)', flu_file_drive),
]

flu_df = None

# Try loading existing file (prioritize CDC for US)
for source_name, flu_file in flu_file_paths:
    if flu_file and os.path.exists(flu_file):
        try:
            flu_df = pd.read_csv(flu_file, parse_dates=['date'] if 'date' in pd.read_csv(flu_file, nrows=1).columns else None)
            if flu_df is None:
                # Try to find date column with different name
                temp_df = pd.read_csv(flu_file, nrows=1)
                date_col = None
                for col in temp_df.columns:
                    if 'date' in col.lower() or 'week' in col.lower() or 'time' in col.lower():
                        date_col = col
                        break
                if date_col:
                    flu_df = pd.read_csv(flu_file, parse_dates=[date_col])
                    flu_df.rename(columns={date_col: 'date'}, inplace=True)
                else:
                    continue

            print(f"✓ Found existing flu data ({source_name}): {flu_file}")
            print(f"  Records: {len(flu_df):,}")
            print(f"  Date range: {flu_df['date'].min().date()} to {flu_df['date'].max().date()}")

            # Check if we need to create flu_activity_level from CDC data
            if 'flu_activity_level' not in flu_df.columns:
                # CDC FluView typically has columns like 'wili', 'ili', 'epiweek' etc.
                # Try to create activity level from available columns
                if 'wili' in flu_df.columns:
                    # WILI (Weighted ILI) - normalize to 0-10 scale
                    # WILI is typically 0-13%, so we'll cap at 10 for activity level
                    flu_df['flu_activity_level'] = flu_df['wili'].apply(
                        lambda x: min(10.0, x) if pd.notna(x) else None
                    )
                    print(f"  ✓ Created flu_activity_level from CDC WILI data")
                elif 'ili' in flu_df.columns:
                    # ILI percentage - normalize to 0-10 scale
                    max_ili = flu_df['ili'].max()
                    if max_ili > 10:
                        flu_df['flu_activity_level'] = (flu_df['ili'] / max_ili) * 10
                    else:
                        flu_df['flu_activity_level'] = flu_df['ili']
                    print(f"  ✓ Created flu_activity_level from CDC ILI data")
                else:
                    print(f"  ⚠️  Note: Could not automatically create flu_activity_level")
                    print(f"     You may need to manually process CDC data")
            else:
                # Check if activity level needs regeneration (old scaling issue)
                max_activity = flu_df['flu_activity_level'].max()
                if max_activity < 1.0:
                    print(f"  ⚠️  Note: Activity levels appear to be on 0-1 scale (max: {max_activity:.2f})")
                    print(f"     This suggests old scaling. Delete the file to regenerate with correct 0-10 scale.")
                else:
                    print(f"  ✓ Activity level range: {flu_df['flu_activity_level'].min():.2f} to {flu_df['flu_activity_level'].max():.2f} (expected: 0-10)")
            break
        except Exception as e:
            print(f"⚠️  Error loading {flu_file}: {e}")

# Option 1: Download from CDC FluView API (recommended for US hospitals)
# CDC FluView API: https://github.com/cmu-delphi/delphi-epidata
# Note: CDC FluView provides weekly flu surveillance data for US states/regions

# Option 2: Use seasonal indicators as proxy (if CDC unavailable)
# Create a seasonal proxy based on typical flu patterns

# Option 3: Skip flu data (models will work without it)

# Try downloading from CDC FluView API
if flu_df is None:
    print("\n📥 Downloading flu data from CDC FluView (Delphi Epidata API)...")
    print(f"  Location: Florida, US (for ADVENTHEALTH ORLANDO)")
    print(f"  Years: {flu_years[0]} to {flu_years[-1]} ({len(flu_years)} years)")

    import requests
    from datetime import datetime, timedelta

    # Delphi API endpoint
    base_url = 'https://api.delphi.cmu.edu/epidata/fluview/'

    all_data = []

    # Region codes for CDC FluView (in order of preference)
    # 1. Florida state code ('fl') - most specific for Orlando
    # 2. HHS Region 4 ('hhs4') - includes FL, GA, AL, KY, MS, NC, SC, TN
    # 3. National ('nat') - fallback if regional data unavailable
    region_codes = ['fl', 'hhs4', 'nat']

    for year in flu_years:
        print(f"\n  Downloading {year} season...")

        # Flu season typically spans two calendar years (Oct of previous year to May of current year)
        # For each year, fetch flu season: Oct (year-1) to May (year)
        season_start = year - 1
        season_end = year

        # Epiweeks: Week 40 of season_start to Week 20 of season_end
        start_epiweek = f"{season_start}40"
        end_epiweek = f"{season_end}20"

        season_data = []
        used_region = None

        # Try each region code until we get data
        for region_code in region_codes:
            try:
                params = {
                    'regions': region_code,
                    'epiweeks': f'{start_epiweek}-{end_epiweek}'
                }

                region_names = {'fl': 'Florida', 'hhs4': 'HHS Region 4', 'nat': 'National'}
                print(f"    Trying {region_names.get(region_code, region_code)} ({start_epiweek}-{end_epiweek})...", end=" ")

                response = requests.get(base_url, params=params, timeout=30)

                if response.status_code == 200:
                    result = response.json()

                    if result.get('result') == 1 and 'epidata' in result:
                        episodes = result['epidata']

                        if episodes:
                            for episode in episodes:
                                # Extract relevant fields
                                epiweek = episode.get('epiweek')
                                wili = episode.get('wili', None)  # Weighted ILI percentage (0-10 scale, can go higher)
                                ili = episode.get('ili', None)    # ILI percentage (0-10 scale)

                                # Convert epiweek to date (approximate - epiweek is YYYYWW format)
                                if epiweek:
                                    year_part = int(str(epiweek)[:4])
                                    week_part = int(str(epiweek)[4:])

                                    # CDC epiweeks run Sunday to Saturday
                                    # Hospital data collection_week is the Sunday (start of week)
                                    # So we need the SUNDAY of each epiweek to match hospital dates
                                    jan1 = datetime(year_part, 1, 1)
                                    # Find first Sunday of year (epiweeks start on Sunday)
                                    days_until_sunday = (6 - jan1.weekday()) % 7
                                    first_sunday = jan1 + timedelta(days=days_until_sunday)
                                    # Use Sunday (START of epiweek) to match hospital collection_week
                                    episode_date = first_sunday + timedelta(weeks=week_part-1)

                                    # Convert wili to activity level (0-10 scale)
                                    # WILI from CDC API is already a percentage (0-10 scale, can exceed 10 during severe seasons)
                                    # CDC activity levels: 0-2.4=Minimal, 2.5-4.9=Low, 5.0-7.4=Moderate, 7.5-9.9=High, 10+=Very High
                                    # Use WILI directly as activity level (it's already on the correct scale)
                                    if wili is not None:
                                        # WILI is already on 0-10 scale, just cap at 10.0 for consistency
                                        activity_level = min(10.0, wili)
                                    elif ili is not None:
                                        # ILI is also on 0-10 scale
                                        activity_level = min(10.0, ili)
                                    else:
                                        activity_level = None

                                    # Flu season indicator (Nov-Feb typically peak season)
                                    month = episode_date.month
                                    is_flu_season = 1 if month in [11, 12, 1, 2] else 0

                                    season_data.append({
                                        'date': episode_date,
                                        'epiweek': epiweek,
                                        'flu_activity_level': round(activity_level, 2) if activity_level else None,
                                        'wili': wili,  # Weighted ILI percentage
                                        'ili': ili,     # ILI percentage
                                        'is_flu_season': is_flu_season,
                                        'region': region_code
                                    })

                            print(f"✓ {len(episodes)} weeks")
                            used_region = region_code
                            break  # Got data, no need to try other regions
                        else:
                            print(f"no data")
                    else:
                        error_msg = result.get('message', 'no results')
                        print(f"{error_msg}")
                else:
                    print(f"HTTP {response.status_code}")

            except Exception as e:
                print(f"error: {e}")

        # Add season data to all_data
        if season_data:
            all_data.extend(season_data)
            if used_region != 'fl':
                print(f"    ℹ️  Using {region_names.get(used_region, used_region)} data for {year} season (Florida data not available)")
        else:
            print(f"    ⚠️  No CDC data available for {year} season from any region")
            print(f"    Note: Will use seasonal proxy for missing dates")

    if all_data:
        flu_df = pd.DataFrame(all_data)

        # Remove duplicates (keep first)
        flu_df = flu_df.drop_duplicates(subset=['date'], keep='first')

        # Sort by date
        flu_df = flu_df.sort_values('date').reset_index(drop=True)

        print(f"\n✓ Downloaded {len(flu_df):,} flu data records from CDC API")
        print(f"  Date range: {flu_df['date'].min().date()} to {flu_df['date'].max().date()}")
        print(f"  Note: Missing dates will be filled with seasonal proxy")

        # DIAGNOSTIC: Verify day of week for CDC flu dates
        first_flu_date = flu_df['date'].min()
        last_flu_date = flu_df['date'].max()
        print(f"\n  📊 CDC FLU DATA DIAGNOSTICS:")
        print(f"     First date: {first_flu_date.date()} ({first_flu_date.strftime('%A')})")
        print(f"     Last date: {last_flu_date.date()} ({last_flu_date.strftime('%A')})")
        if first_flu_date.strftime('%A') != 'Sunday':
            print(f"     ⚠️  WARNING: Flu dates are NOT Sundays! May cause alignment issues with hospital data.")
        else:
            print(f"     ✓ Flu dates are Sundays (matches hospital collection_week)")
    else:
        print("\n⚠️  No flu data downloaded from CDC API")
        print("   Note: This may be normal if API is temporarily unavailable or for early years")
        print("   All dates will use seasonal proxy based on month patterns")
        print("   You can download manually from: https://www.cdc.gov/flu/weekly/fluactivitysurv.htm")

# Clean and validate flu data
if flu_df is not None and len(flu_df) > 0:
    print("\n🧹 Cleaning flu data...")

    initial_count = len(flu_df)

    # Remove duplicates
    flu_df = flu_df.drop_duplicates(subset=['date']).reset_index(drop=True)

    # Ensure date column is datetime
    flu_df['date'] = pd.to_datetime(flu_df['date'])

    # Remove invalid dates
    flu_df = flu_df[flu_df['date'].notna()]

    # Sort by date
    flu_df = flu_df.sort_values('date').reset_index(drop=True)

    # Regenerate activity_level from WILI if needed (fixes old scaling issue)
    # Check if activity_level needs regeneration (old scaling was 0-1, new should be 0-10)
    needs_regeneration = False
    if 'flu_activity_level' in flu_df.columns and 'wili' in flu_df.columns:
        max_activity = flu_df['flu_activity_level'].max()
        if max_activity < 1.0:
            # Old scaling detected - regenerate from WILI
            needs_regeneration = True
            print(f"  🔧 Regenerating activity_level from WILI (fixing old 0-1 scale)")

    if needs_regeneration and 'wili' in flu_df.columns:
        # Regenerate activity_level from WILI using correct scaling
        flu_df['flu_activity_level'] = flu_df['wili'].apply(
            lambda x: min(10.0, x) if pd.notna(x) else None
        )
        print(f"  ✓ Activity level regenerated: {flu_df['flu_activity_level'].min():.2f} to {flu_df['flu_activity_level'].max():.2f}")

    # Fill missing values
    if 'flu_activity_level' in flu_df.columns:
        flu_df['flu_activity_level'] = flu_df['flu_activity_level'].ffill().bfill()

    if 'is_flu_season' in flu_df.columns:
        flu_df['is_flu_season'] = flu_df['is_flu_season'].fillna(0).astype(int)

    # Validate required columns
    required_cols = ['date', 'flu_activity_level']
    missing_cols = [col for col in required_cols if col not in flu_df.columns]

    if missing_cols:
        print(f"⚠️  Missing required columns: {missing_cols}")
        flu_df = None
    else:
        print(f"✓ Flu data cleaned: {len(flu_df):,} records (removed {initial_count - len(flu_df)} duplicates/invalid)")
        print(f"  Date range: {flu_df['date'].min().date()} to {flu_df['date'].max().date()}")
        print(f"  Columns: {', '.join(flu_df.columns)}")
        print(f"  Note: This is weekly data (matching bed occupancy frequency)")

        # Ensure flu data covers full hospital date range with seasonal proxy
        print(f"\n📅 Aligning flu data to hospital date range...")

        # Get hospital dates for alignment (must have occupancy_clean)
        hospital_weekly_dates = None
        if 'occupancy_clean' in globals() and len(occupancy_clean) > 0 and 'aligned_date' in occupancy_clean.columns:
            hospital_start = occupancy_clean['aligned_date'].min()
            hospital_end = occupancy_clean['aligned_date'].max()

            if not pd.isna(hospital_start) and not pd.isna(hospital_end):
                print(f"  Hospital date range: {hospital_start.date()} to {hospital_end.date()}")
                # Use actual hospital dates (not a generated range) to ensure exact match
                hospital_weekly_dates = pd.to_datetime(occupancy_clean['aligned_date']).unique()
                hospital_weekly_dates = pd.DatetimeIndex(sorted(hospital_weekly_dates))

        if hospital_weekly_dates is not None and len(hospital_weekly_dates) > 0:
            # Start with CDC data if available - align to hospital dates
            if len(flu_df) > 0:
                flu_aligned = align_data_to_dates(flu_df, hospital_weekly_dates, tolerance_days=3)
            else:
                flu_aligned = pd.DataFrame()

            # Create complete flu dataframe for all hospital weeks
            # Fill missing dates with seasonal proxy
            flu_complete_data = []
            flu_indexed = flu_aligned if len(flu_aligned) > 0 else pd.DataFrame()

            for date in hospital_weekly_dates:
                if len(flu_indexed) > 0 and date in flu_indexed.index:
                    # Use aligned CDC data, but check if it has actual values
                    row = flu_indexed.loc[date].to_dict()
                    # Check if CDC data has actual values (not all NaN)
                    has_actual_data = (pd.notna(row.get('wili')) or pd.notna(row.get('ili')))

                    if has_actual_data:
                        # Use CDC data
                        row['date'] = date
                        row['is_proxy'] = 0
                        flu_complete_data.append(row)
                    else:
                        # CDC data exists but is all NaN - use seasonal proxy instead
                        # Fill all flu columns with seasonal proxy to avoid NaN gaps
                        seasonal_value = get_seasonal_flu_activity(date)
                        # Calculate epiweek from date for consistency
                        year = date.year
                        jan1 = datetime(year, 1, 1)
                        days_until_sunday = (6 - jan1.weekday()) % 7
                        first_sunday = jan1 + timedelta(days=days_until_sunday)
                        week_num = ((date - first_sunday).days // 7) + 1
                        epiweek_proxy = int(f"{year}{week_num:02d}")

                        flu_complete_data.append({
                            'date': date,
                            'epiweek': epiweek_proxy,  # Calculate from date for consistency
                            'flu_activity_level': seasonal_value,
                            'wili': seasonal_value,  # Fill with proxy to avoid NaN
                            'ili': seasonal_value,   # Fill with proxy to avoid NaN
                            'is_flu_season': 1 if date.month in [11, 12, 1, 2] else 0,
                            'region': 'proxy',  # Mark as proxy for consistency
                            'is_proxy': 1
                        })
                else:
                    # No CDC data for this date - use seasonal proxy
                    # Fill all flu columns with seasonal proxy to avoid NaN gaps in feature engineering
                    seasonal_value = get_seasonal_flu_activity(date)
                    # Calculate epiweek from date for consistency
                    year = date.year
                    jan1 = datetime(year, 1, 1)
                    days_until_sunday = (6 - jan1.weekday()) % 7
                    first_sunday = jan1 + timedelta(days=days_until_sunday)
                    week_num = ((date - first_sunday).days // 7) + 1
                    epiweek_proxy = int(f"{year}{week_num:02d}")

                    flu_complete_data.append({
                        'date': date,
                        'epiweek': epiweek_proxy,  # Calculate from date for consistency
                        'flu_activity_level': seasonal_value,
                        'wili': seasonal_value,  # Fill with proxy to avoid NaN
                        'ili': seasonal_value,   # Fill with proxy to avoid NaN
                        'is_flu_season': 1 if date.month in [11, 12, 1, 2] else 0,
                        'region': 'proxy',  # Mark as proxy for consistency
                        'is_proxy': 1
                    })

            flu_df = pd.DataFrame(flu_complete_data).sort_values('date').reset_index(drop=True)

            # Ensure is_proxy column exists and is correctly set
            # CDC data should have wili/ili not null and is_proxy=0
            if 'is_proxy' not in flu_df.columns:
                flu_df['is_proxy'] = 1  # Default to proxy
            # Mark CDC data (has wili/ili) as non-proxy
            # Check region column to distinguish CDC data from proxy data
            # Proxy data has region='proxy', CDC data has region in ['fl', 'hhs4', 'nat']
            if 'region' in flu_df.columns:
                # CDC regions are actual data sources, proxy is synthetic
                cdc_regions = ['fl', 'hhs4', 'nat']
                has_cdc_data = flu_df['region'].isin(cdc_regions)
                flu_df.loc[has_cdc_data, 'is_proxy'] = 0
                # Ensure proxy rows are marked correctly
                flu_df.loc[flu_df['region'] == 'proxy', 'is_proxy'] = 1
            else:
                # Fallback: check if wili/ili came from CDC (not just filled)
                # This is less reliable but better than nothing
                has_cdc_data = flu_df['wili'].notna() | flu_df['ili'].notna()
                flu_df.loc[has_cdc_data, 'is_proxy'] = 0

            # Ensure all rows have flu_activity_level (fill any remaining NaN with seasonal proxy)
            # This handles edge cases where flu_activity_level might be missing
            missing_activity = flu_df['flu_activity_level'].isna()
            if missing_activity.sum() > 0:
                for idx in flu_df[missing_activity].index:
                    date = flu_df.loc[idx, 'date']
                    flu_df.loc[idx, 'flu_activity_level'] = get_seasonal_flu_activity(date)
                    flu_df.loc[idx, 'is_flu_season'] = 1 if date.month in [11, 12, 1, 2] else 0
                    flu_df.loc[idx, 'is_proxy'] = 1

            # Verify dates match exactly
            flu_dates = pd.to_datetime(flu_df['date']).unique()
            if len(flu_dates) == len(hospital_weekly_dates) and all(d in hospital_weekly_dates for d in flu_dates):
                print(f"  ✓ Flu data aligned to {len(hospital_weekly_dates)} hospital dates (exact match)")
            else:
                print(f"  ⚠️  Warning: Flu dates may not match hospital dates exactly")

            # Additional validation
            cdc_weeks = (flu_df['is_proxy'] == 0).sum() if 'is_proxy' in flu_df.columns else 0
            proxy_weeks = (flu_df['is_proxy'] == 1).sum() if 'is_proxy' in flu_df.columns else len(flu_df)

            print(f"  CDC data: {cdc_weeks} weeks, Seasonal proxy: {proxy_weeks} weeks")
            print(f"  Flu date range: {flu_df['date'].min().date()} to {flu_df['date'].max().date()}")
            print(f"  Hospital date range: {hospital_weekly_dates.min().date()} to {hospital_weekly_dates.max().date()}")

            # Check for any issues
            if cdc_weeks == 0 and len(flu_df) > 0 and 'wili' in flu_df.columns and flu_df['wili'].notna().sum() > 0:
                print(f"  ⚠️  Note: No CDC data matched (all proxy). CDC data may have different date format.")
            elif cdc_weeks > 0:
                print(f"  ✓ Flu data fully covers hospital date range ({cdc_weeks} CDC weeks, {proxy_weeks} proxy weeks)")
            else:
                print(f"  ✓ Flu data fully covers hospital date range (all seasonal proxy)")

            # Verify all flu columns are complete (no NaN values)
            flu_columns_to_check = ['flu_activity_level', 'wili', 'ili', 'epiweek', 'region', 'is_flu_season']
            flu_columns_to_check = [col for col in flu_columns_to_check if col in flu_df.columns]
            flu_missing = flu_df[flu_columns_to_check].isna().sum()
            if flu_missing.sum() == 0:
                print(f"  ✓ All flu columns complete (wili/ili/epiweek/region filled with seasonal proxy for off-season weeks)")
            else:
                print(f"  ⚠️  Some flu columns have missing values: {flu_missing.to_dict()}")
        else:
            print(f"  ⚠️  Warning: Hospital dates not available - flu data not aligned")
            print(f"     Flu data will be saved as-is (can be aligned later)")

        # Save cleaned flu data (save to both locations)
        save_to_locations(flu_df, 'flu_cdc.csv', DATA_EXTERNAL, DRIVE_DATA_EXTERNAL)
else:
    print("\n⚠️  No flu data available")
    print("   Options:")
    print("   1. Download from CDC FluView API (automatic, see code above)")
    print("   2. Download manually from: https://www.cdc.gov/flu/weekly/fluactivitysurv.htm")
    print("   3. Use seasonal proxy (created from date patterns)")
    print("   Save as: data/external/flu_cdc.csv")


FLU DATA EXTRACTION (Florida, US)

📋 Options for Florida flu data:
   1. CDC FluView API (US data, includes Florida)
   2. Use seasonal indicators as proxy (if CDC unavailable)
   3. Skip flu data (models work without it)

✅ Note: CDC FluView is perfect for US hospitals like ADVENTHEALTH ORLANDO in Florida.
  Using hospital date range: 2020-07-19 to 2024-04-21
  Flu years to fetch: [2020, 2021, 2022, 2023, 2024]
✓ Found existing flu data (CDC FluView): /content/hospital_occupancy_forecasting/data/external/flu_cdc.csv
  Records: 197
  Date range: 2020-07-19 to 2024-04-21
  ✓ Activity level range: 0.50 to 6.88 (expected: 0-10)

🧹 Cleaning flu data...
✓ Flu data cleaned: 197 records (removed 0 duplicates/invalid)
  Date range: 2020-07-19 to 2024-04-21
  Columns: date, epiweek, flu_activity_level, wili, ili, is_flu_season, region, is_proxy
  Note: This is weekly data (matching bed occupancy frequency)

📅 Aligning flu data to hospital date range...
  Hospital date range: 2020-07-19 to 2024

# Part 3: Verification & Saving

Verify all extracted and processed data, then save cleaned files for the next notebook.


### 3.1 Verify All Data

Verify all extracted and processed data: external data (weather, flu) and cleaned data with aligned dates.

**Note:** This section reuses the `weather_df` and `flu_df` variables from extraction if available, otherwise loads from saved files.


In [27]:
# ============================================================
# VERIFY WEATHER DATA
# ============================================================

print("="*60)
print("WEATHER DATA SAMPLE")
print("="*60)

# Reuse weather_df from extraction if available, otherwise load from file
if 'weather_df' in globals() and weather_df is not None and len(weather_df) > 0:
    weather_sample = weather_df.copy()
    print(f"\n✓ Using weather data from extraction (in memory)")
else:
    weather_file_paths = [
        os.path.join(DRIVE_DATA_EXTERNAL, 'weather_orlando.csv') if DRIVE_DATA_EXTERNAL else None,
        os.path.join(DATA_EXTERNAL, 'weather_orlando.csv')
    ]
    weather_file_paths = [p for p in weather_file_paths if p]  # Remove None values
    weather_sample = None
    for file_path in weather_file_paths:
        if os.path.exists(file_path):
            try:
                weather_sample = pd.read_csv(file_path, parse_dates=['date'])
                print(f"\n✓ Loaded from: {file_path}")
                break
            except Exception as e:
                continue

if weather_sample is not None and len(weather_sample) > 0:
    print(f"\nFirst 10 rows of weather data:")
    print(weather_sample.head(10).to_string())

    print(f"\n\nLast 10 rows of weather data:")
    print(weather_sample.tail(10).to_string())

    print(f"\n\nWeather Data Statistics:")
    print(weather_sample.describe())

    print(f"\n\nDate Range Check:")
    print(f"  First date: {weather_sample['date'].min()}")
    print(f"  Last date: {weather_sample['date'].max()}")
    print(f"  Total records: {len(weather_sample)}")
    print(f"  Data frequency: Weekly (aggregated from daily)")
    print(f"  Expected weeks (2020-2024): ~214")
    # Check for missing weeks
    if len(weather_sample) > 0:
        weekly_range = pd.date_range(
            start=weather_sample['date'].min(),
            end=weather_sample['date'].max(),
            freq='W-SUN'  # Week ending Sunday (matching hospital data)
        )
        missing_weeks = len(weekly_range) - len(weather_sample)
        if missing_weeks > 0:
            print(f"  Missing weeks: {missing_weeks}")
        else:
            print(f"  Missing weeks: 0 (complete)")

    print(f"\n\nTemperature Range Check:")
    if 'temp_avg' in weather_sample.columns:
        print(f"  Average temp range: {weather_sample['temp_avg'].min():.1f}°F to {weather_sample['temp_avg'].max():.1f}°F")
        print(f"  (Expected for Orlando, FL: ~40°F to ~95°F)")
    if 'precipitation' in weather_sample.columns:
        print(f"  Precipitation range: {weather_sample['precipitation'].min():.2f} to {weather_sample['precipitation'].max():.2f} inches")
else:
    print("\n⚠️  Weather data file not found")

WEATHER DATA SAMPLE

✓ Using weather data from extraction (in memory)

First 10 rows of weather data:
        date   temp_avg   temp_max   temp_min  precipitation  wind_speed     pressure
0 2020-07-19  84.714286  93.534286  78.311429       1.929135   11.442857  1017.642857
1 2020-07-26  82.888571  90.577143  77.437143       0.854331   11.585714  1018.657143
2 2020-08-02  83.145714  92.608571  77.565714       0.641733    9.557143  1017.385714
3 2020-08-09  83.608571  94.254286  76.125714       3.980317    7.742857  1017.271429
4 2020-08-16  83.994286  93.688571  78.774286       1.527560    7.500000  1016.457143
5 2020-08-23  81.705714  90.988571  76.408571       0.606300    9.428571  1014.542857
6 2020-08-30  84.971429  92.582857  80.137143       0.732284   13.242857  1016.342857
7 2020-09-06  84.071429  93.328571  78.851429       1.629922    8.785714  1016.942857
8 2020-09-13  80.908571  89.600000  77.128571       5.094491   10.828571  1013.828571
9 2020-09-20  82.014286  89.857143  77

In [28]:
# ============================================================
# VERIFY FLU DATA
# ============================================================

print("="*60)
print("FLU DATA SAMPLE")
print("="*60)

# Reuse flu_df from extraction if available, otherwise load from file
if 'flu_df' in globals() and flu_df is not None and len(flu_df) > 0:
    flu_sample = flu_df.copy()
    print(f"\n✓ Using flu data from extraction (in memory)")
else:
    flu_file_paths = [
        os.path.join(DRIVE_DATA_EXTERNAL, 'flu_cdc.csv'),
        os.path.join(DATA_EXTERNAL, 'flu_cdc.csv')
    ]
    flu_sample = None
    for file_path in flu_file_paths:
        if os.path.exists(file_path):
            try:
                flu_sample = pd.read_csv(file_path, parse_dates=['date'])
                print(f"\n✓ Loaded from: {file_path}")
                break
            except Exception as e:
                continue

if flu_sample is not None and len(flu_sample) > 0:
    print(f"\nFirst 10 rows of flu data:")
    print(flu_sample.head(10).to_string())

    print(f"\n\nLast 10 rows of flu data:")
    print(flu_sample.tail(10).to_string())

    print(f"\n\nFlu Data Statistics:")
    if 'flu_activity_level' in flu_sample.columns:
        print(flu_sample[['flu_activity_level', 'wili', 'ili', 'is_flu_season']].describe())

    print(f"\n\nDate Range Check:")
    print(f"  First date: {flu_sample['date'].min()}")
    print(f"  Last date: {flu_sample['date'].max()}")
    print(f"  Total records: {len(flu_sample)}")
    print(f"  Expected records (weekly, 5 seasons): ~165")
    print(f"  Note: Flu data is for Florida, US (appropriate for ADVENTHEALTH ORLANDO)")

    print(f"\n\nFlu Season Check:")
    if 'is_flu_season' in flu_sample.columns:
        flu_season_count = flu_sample['is_flu_season'].sum()
        print(f"  Records marked as flu season (Nov-Feb): {flu_season_count} ({flu_season_count/len(flu_sample)*100:.1f}%)")
        print(f"  Expected: ~40-50% (4 months out of 12)")

    print(f"\n\nActivity Level Check:")
    if 'flu_activity_level' in flu_sample.columns:
        print(f"  Activity level range: {flu_sample['flu_activity_level'].min():.2f} to {flu_sample['flu_activity_level'].max():.2f}")
        print(f"  Expected range: 0-10")
        print(f"  Mean activity: {flu_sample['flu_activity_level'].mean():.2f}")
else:
    print("\n⚠️  Flu data file not found")

FLU DATA SAMPLE

✓ Using flu data from extraction (in memory)

First 10 rows of flu data:
    epiweek  flu_activity_level  wili  ili  is_flu_season region  is_proxy       date
0  202029.0                 0.5   0.5  0.5              0  proxy         1 2020-07-19
1  202030.0                 0.5   0.5  0.5              0  proxy         1 2020-07-26
2  202031.0                 0.5   0.5  0.5              0  proxy         1 2020-08-02
3  202032.0                 0.5   0.5  0.5              0  proxy         1 2020-08-09
4  202033.0                 0.5   0.5  0.5              0  proxy         1 2020-08-16
5  202034.0                 0.5   0.5  0.5              0  proxy         1 2020-08-23
6  202035.0                 0.5   0.5  0.5              0  proxy         1 2020-08-30
7  202036.0                 1.0   1.0  1.0              0  proxy         1 2020-09-06
8  202037.0                 1.0   1.0  1.0              0  proxy         1 2020-09-13
9  202038.0                 1.0   1.0  1.0        

In [29]:
# ============================================================
# VERIFY CLEANED OCCUPANCY DATA
# ============================================================

print("="*60)
print("CLEANED Occupancy DATA SAMPLE")
print("="*60)

if len(occupancy_clean) > 0:
    print(f"\nFirst 10 rows of cleaned bed occupancy data:")
    # Show key columns for verification
    key_cols = ['date', 'aligned_date', 'bed_occupancy', 'bed_capacity',
                'occupancy_rate', 'hospital_name']
    display_cols = [col for col in key_cols if col in occupancy_clean.columns]
    print(occupancy_clean[display_cols].head(10).to_string())

    print(f"\n\nSample showing dates (COVID data uses real dates, no alignment needed):")
    sample_aligned = occupancy_clean[['aligned_date']].head(10)
    for idx, row in sample_aligned.iterrows():
        aligned_date_val = pd.to_datetime(row['aligned_date']).strftime('%Y-%m-%d')
        print(f"  {aligned_date_val}")

    print(f"\n\nBed Occupancy Data Statistics:")
    print(f"  Total records: {len(occupancy_clean):,}")
    print(f"  Date range: {occupancy_clean['aligned_date'].min()} to {occupancy_clean['aligned_date'].max()}")
    print(f"  Note: aligned_date is the collection week date (no transformation needed for COVID data)")

    if 'bed_occupancy' in occupancy_clean.columns:
        occupancy = occupancy_clean['bed_occupancy'].dropna()
        print(f"  Bed Occupancy: Mean {occupancy.mean():.1f} beds, Median {occupancy.median():.1f} beds, Range {occupancy.min():.1f}-{occupancy.max():.1f} beds")

    if 'occupancy_rate' in occupancy_clean.columns:
        rate = occupancy_clean['occupancy_rate'].dropna()
        print(f"  Occupancy Rate: Mean {rate.mean():.1f}%, Median {rate.median():.1f}%, Range {rate.min():.1f}%-{rate.max():.1f}%")

    print(f"\n\nData Structure Notes:")
    print(f"  • aligned_date: Collection week date (real dates, no transformation)")
    print(f"  • bed_occupancy: Weekly 7-day averages (weekly frequency)")
    print(f"  • is_imputed: Indicator for weeks that were interpolated (if any)")
else:
    print("\n⚠️  No cleaned data available")

CLEANED Occupancy DATA SAMPLE

First 10 rows of cleaned bed occupancy data:
        date aligned_date  bed_occupancy  bed_capacity  occupancy_rate         hospital_name
0 2020-07-19   2020-07-19         2011.0        2825.7       71.168206  ADVENTHEALTH ORLANDO
1 2020-07-26   2020-07-26         2022.9        2823.3       71.650197  ADVENTHEALTH ORLANDO
2 2020-08-02   2020-08-02         2007.7        2821.0       71.169798  ADVENTHEALTH ORLANDO
3 2020-08-09   2020-08-09         1907.6        2654.3       71.868289  ADVENTHEALTH ORLANDO
4 2020-08-16   2020-08-16         2015.4        2814.6       71.605201  ADVENTHEALTH ORLANDO
5 2020-08-23   2020-08-23         2000.4        2816.9       71.014236  ADVENTHEALTH ORLANDO
6 2020-08-30   2020-08-30         1916.3        2811.0       68.171469  ADVENTHEALTH ORLANDO
7 2020-09-06   2020-09-06         1882.0        2817.1       66.806290  ADVENTHEALTH ORLANDO
8 2020-09-13   2020-09-13         1919.1        2834.4       67.707451  ADVENTHEALTH OR

In [30]:
# Save cleaned data WITH aligned_date column
print(f"\n💾 Saving cleaned data...")

if len(occupancy_clean) == 0:
    raise ValueError("❌ No cleaned data to save! Please check data extraction steps.")

# Verify required columns exist
has_aligned_date = 'aligned_date' in occupancy_clean.columns
# Removed: has_admittime (using aligned_date only) occupancy_clean.columns
has_bed_occupancy = 'bed_occupancy' in occupancy_clean.columns

if not has_aligned_date:
    print("⚠️  WARNING: Missing 'aligned_date' column - creating aligned_date...")
    raise ValueError("❌ Missing 'aligned_date' columns!")

if not has_bed_occupancy:
    raise ValueError("❌ Missing 'bed_occupancy' column - required for forecasting!")

# Save to all locations
saved_file = save_to_locations(occupancy_clean, 'occupancy_clean.csv', DATA_RAW, DRIVE_DATA_RAW)
file_size = os.path.getsize(saved_file) / (1024**2)
print(f"    Size: {file_size:.2f} MB, Records: {len(occupancy_clean):,}")

# Verify saved file
try:
    verify_df = pd.read_csv(saved_file, nrows=5)
    print(f"\n✓ Verification: Successfully read saved file")
    print(f"  Columns: {list(verify_df.columns)}")
except Exception as e:
    print(f"  ⚠️  Warning: Could not verify saved file: {e}")

# Summary
print(f"\n📋 Data Summary:")
print(f"  ✓ Total records: {len(occupancy_clean):,}")
print(f"  ✓ Date range: {occupancy_clean['aligned_date'].min().date()} to {occupancy_clean['aligned_date'].max().date()}")
print(f"  ✓ aligned_date: {occupancy_clean['aligned_date'].notna().sum():,} non-null (100%)")
print(f"  ✓ bed_occupancy: {occupancy_clean['bed_occupancy'].notna().sum():,} non-null ({occupancy_clean['bed_occupancy'].notna().sum()/len(occupancy_clean)*100:.1f}%)")

# ============================================================
# FINAL DATE ALIGNMENT VERIFICATION
# ============================================================
print(f"\n" + "="*60)
print("FINAL DATE ALIGNMENT VERIFICATION")
print("="*60)

# Get hospital dates
hospital_dates = pd.to_datetime(occupancy_clean['aligned_date']).sort_values().unique()
print(f"\n📊 Hospital Data (occupancy_clean):")
print(f"   Records: {len(hospital_dates)}")
print(f"   Date range: {hospital_dates.min().date()} to {hospital_dates.max().date()}")
print(f"   First date day of week: {hospital_dates[0].strftime('%A')}")
print(f"   Last date day of week: {hospital_dates[-1].strftime('%A')}")

# Verify all hospital dates are Sundays
non_sunday_hospital = sum(1 for d in hospital_dates if d.strftime('%A') != 'Sunday')
if non_sunday_hospital > 0:
    print(f"   ⚠️  WARNING: {non_sunday_hospital} hospital dates are NOT Sundays!")
else:
    print(f"   ✓ All hospital dates are Sundays")

# Check weather alignment
if 'weather_df' in globals() and weather_df is not None and len(weather_df) > 0:
    weather_dates = pd.to_datetime(weather_df['date']).sort_values().unique()
    print(f"\n📊 Weather Data (weather_df):")
    print(f"   Records: {len(weather_dates)}")
    print(f"   Date range: {weather_dates.min().date()} to {weather_dates.max().date()}")
    print(f"   First date day of week: {weather_dates[0].strftime('%A')}")

    # Check alignment with hospital
    weather_set = set(weather_dates)
    hospital_set = set(hospital_dates)

    exact_match = len(weather_set & hospital_set)
    weather_only = len(weather_set - hospital_set)
    hospital_only = len(hospital_set - weather_set)

    print(f"   Exact date matches with hospital: {exact_match}/{len(hospital_dates)} ({exact_match/len(hospital_dates)*100:.1f}%)")
    if weather_only > 0:
        print(f"   ⚠️  Weather dates not in hospital: {weather_only}")
    if hospital_only > 0:
        print(f"   ⚠️  Hospital dates not in weather: {hospital_only}")
    if exact_match == len(hospital_dates):
        print(f"   ✓ PERFECT ALIGNMENT: Weather dates match all hospital dates")
else:
    print(f"\n📊 Weather Data: Not available")

# Check flu alignment
if 'flu_df' in globals() and flu_df is not None and len(flu_df) > 0:
    flu_dates = pd.to_datetime(flu_df['date']).sort_values().unique()
    print(f"\n📊 Flu Data (flu_df):")
    print(f"   Records: {len(flu_dates)}")
    print(f"   Date range: {flu_dates.min().date()} to {flu_dates.max().date()}")
    print(f"   First date day of week: {flu_dates[0].strftime('%A')}")

    # Check alignment with hospital
    flu_set = set(flu_dates)
    hospital_set = set(hospital_dates)

    exact_match = len(flu_set & hospital_set)
    flu_only = len(flu_set - hospital_set)
    hospital_only = len(hospital_set - flu_set)

    print(f"   Exact date matches with hospital: {exact_match}/{len(hospital_dates)} ({exact_match/len(hospital_dates)*100:.1f}%)")
    if flu_only > 0:
        print(f"   ⚠️  Flu dates not in hospital: {flu_only}")
    if hospital_only > 0:
        print(f"   ⚠️  Hospital dates not in flu: {hospital_only}")
    if exact_match == len(hospital_dates):
        print(f"   ✓ PERFECT ALIGNMENT: Flu dates match all hospital dates")
else:
    print(f"\n📊 Flu Data: Not available")

print(f"\n" + "="*60)
print(f"✅ Data extraction complete! Ready for next notebook (01b_data_preprocessing.ipynb)")


💾 Saving cleaned data...
✓ Saved to: /content/hospital_occupancy_forecasting/data/raw/occupancy_clean.csv
✓ Saved to: /content/drive/MyDrive/hospital_occupancy_forecasting/data/raw/occupancy_clean.csv
    Size: 0.03 MB, Records: 197

✓ Verification: Successfully read saved file
  Columns: ['date', 'aligned_date', 'bed_occupancy', 'bed_capacity', 'occupancy_rate', 'coverage', 'hospital_name', 'city', 'state', 'is_imputed', 'icu_occupied', 'icu_capacity', 'covid_inpatients', 'covid_icu', 'staffed_beds', 'icu_occupancy_rate', 'covid_patient_pct']

📋 Data Summary:
  ✓ Total records: 197
  ✓ Date range: 2020-07-19 to 2024-04-21
  ✓ aligned_date: 197 non-null (100%)
  ✓ bed_occupancy: 197 non-null (100.0%)

FINAL DATE ALIGNMENT VERIFICATION

📊 Hospital Data (occupancy_clean):
   Records: 197
   Date range: 2020-07-19 to 2024-04-21
   First date day of week: Sunday
   Last date day of week: Sunday
   ✓ All hospital dates are Sundays

📊 Weather Data (weather_df):
   Records: 197
   Date range

## Summary

This notebook extracts and prepares bed occupancy data from **COVID-19 Reported Patient Impact and Hospital Capacity by Facility** dataset for **ADVENTHEALTH ORLANDO**. Key outputs:
- Cleaned weekly bed occupancy records (weekly frequency maintained)
- **`aligned_date` column**: Collection week date (no alignment needed for COVID data)
- **`is_imputed` column**: Indicator for weeks that were interpolated (if any)
- External data: Weather (Orlando, FL) and Flu (Florida, US) for hospital's date range
- Data quality documentation
- Ready for preprocessing in next notebook

**Output Files:**
- `data/raw/occupancy_clean.csv` - Cleaned weekly bed occupancy data (197 records)
- `data/external/weather_orlando.csv` - Weekly weather data for Orlando, FL (aligned to hospital date range)
- `data/external/flu_cdc.csv` - Weekly flu data from CDC FluView (all columns filled with seasonal proxy for off-season weeks - no NaN values: wili, ili, epiweek, region, flu_activity_level)

**Date Ranges:**
- **Quality data period**: 2020-07-19 to 2024-04-21 (197 weeks with valid occupancy data)
- **Excluded weeks**: 2020-03-22 to 2020-07-12 (17 weeks excluded due to missing occupancy data during early pandemic)
- **External data**: Aligned to hospital quality data date range (2020-07-19 to 2024-04-21)

**Approach:** Data is already weekly (7-day averages). Only weeks with valid occupancy data are included. Data remains at weekly frequency for weekly forecasting models. No anchor year alignment needed since dates are real (not deidentified).

**Data Source:** HealthData.gov - COVID-19 Reported Patient Impact and Hospital Capacity by Facility

**Hospital:** ADVENTHEALTH ORLANDO, Orlando, FL

**Next Steps:** Proceed to `01b_data_preprocessing.ipynb` to further process the time series data for forecasting.
